# Sensitivity charts

## Mappings from dashboard json exports to chart labels and number formatting

In [87]:
import json
from pathlib import Path

# Per-metric display format (string templates)
format_val = {
  # Envelope (spread)
  "median_W":           "{:.1f}",     # absolute width (units of variable)
  "p90_W":              "{:.1f}",
  "RW_med":             "{:.0%}",     # relative width (median)
  "RW_high":            "{:.0%}",     # on event days
  "RW_low":             "{:.0%}",
  "Peak_ratio":         "{:.2f}",

  # Central shift (displacement)
  "CS_abs":             "{:.1f}",     # absolute shift
  "CS_rel":             "{:.1%}",     # relative shift
  "Signed_C_shift":     "{:+.1%}",    # signed relative shift

  # Coverage
  "Coverage_frac":      "{:.0%}",
  "coverage100":        "{:.0%}",
  "Outside_upper":      "{:.0%}",
  "Outside_lower":      "{:.0%}",

  # Event contrast
  "Event_contrast":     "{:+.0%}",    # RW_high − RW_low (relative units)
  "Event_ratio":        "{:.2f}",

  # Duration-curve signature
  "RA_5":               "{:.0%}",
  "RA_50":              "{:.0%}",
  "RA_95":              "{:.0%}",

  # Optional shift detail (from your table)
  "delta_mean":         "{:+.0f}",
  "median_delta":       "{:+.0f}",
  "median_delta_pct":   "{:+.0%}",
  "delta_min":          "{:+.0f}",
  "delta_p50":          "{:+.0f}",
  "delta_p90":          "{:+.0f}",
  "delta_max":          "{:+.0f}",
  "delta_event_pairs_mean": "{:+.0f}",
  "delta_non_event_pairs_mean": "{:+.0f}",
  "normalized_mean_delta": "{:+.0%}",

  # Bias metrics
  "PBIAS%": "{:+.0f}%",
  "Bias(obs-pred)": "{:+.0f}",

  # Alias/legacy keys (if used)
  "relative_width":      "{:.0%}",
  "relative_width_high": "{:.0%}",
  "relative_width_low":  "{:.0%}",
  "relative_width_event": "{:.0%}",
  "relative_width_nonevent": "{:.0%}",
  "relative_width_event_contrast": "{:+.0%}",
  "event_ratio": "{:.2f}×",

  # __baseline__ event/nonevent ensemble spread
  "bl_event_ratio":          "{:.2f}×",
  "bl_RW_event":             "{:.0%}",
  "bl_RW_nonevent":          "{:.0%}",
  "bl_median_W_event":       "{:.1f}",
  "bl_median_W_nonevent":    "{:.1f}",
  "bl_mean_W_event":         "{:.1f}",
  "bl_mean_W_nonevent":      "{:.1f}",
  "bl_p90_W_event":          "{:.1f}",
  "bl_p90_W_nonevent":       "{:.1f}",

  # Absolute width event ratio
  "bl_abs_W_event_ratio":    "{:.1f}×",

  # Mean-based relative width (all / event / non-event)
  "bl_RW_mean_all":          "{:.0%}",
  "bl_RW_mean_event":        "{:.0%}",
  "bl_RW_mean_nonevent":     "{:.0%}",

  # (max-min)/median metrics
  "bl_median_maxmin_over_median":          "{:.0%}",
  "bl_mean_maxmin_over_median":            "{:.0%}",
  "bl_median_maxmin_over_median_event":    "{:.0%}",
  "bl_median_maxmin_over_median_nonevent": "{:.0%}",
  "bl_mean_maxmin_over_median_event":      "{:.0%}",
  "bl_mean_maxmin_over_median_nonevent":   "{:.0%}",
}

# Single source of truth for label selection.
#
# Each key is the technical metric name expected in the exported stats JSON.
# `label` is what you pass in semantic_labels.
# Add `path` only when that metric name appears in multiple places in the export JSON.
# Example: "path": ("overlay_comparison", "__baseline__", "relative_width")
semantic_label_map = {
  # Width & spread
  "median_W": {"label": "Model raw width median (W)"},
  "p90_W": {"label": "Model raw width p90 (W)"},
  "RW_med": {"path": ("overlay_comparison", "__baseline__", "relative_width"), "label": "Model relative width median((p95-p05)/p50)"},
  "RW_high": {"path": ("overlay_comparison", "__baseline__", "relative_width_event"), "label": "Relative width (event days)"},
  "RW_low": {"path": ("overlay_comparison", "__baseline__", "relative_width_nonevent"), "label": "Relative width (non-event days)"},

  # Shift & coverage
  "Coverage_frac": {"label": "Coverage fraction (overlay inside envelope)"},
  "delta_mean": {"path": ("overlay_comparison", "BASE", "delta_mean"), "label": "Mean shift (vs. BASE)"},
  "mean_delta_pct": {"path": ("overlay_comparison", "BASE", "delta_mean_pct"), "label": "Percent of mean shift (vs. BASE)"},
  "median_delta": {"path": ("overlay_comparison", "BASE", "median_delta"), "label": "Median shift (vs. BASE)"},
  "median_delta_pct": {"path": ("overlay_comparison", "BASE", "median_delta_pct"), "label": "Median percent shift (vs. BASE)"},
  "delta_max": {"label": "Shift in max (baseline - overlay)"},
  "delta_event_pairs_mean": {"path": ("overlay_comparison", "BASE", "delta_event_pairs_mean"), "label": "Mean shift events"},
  "delta_non_event_pairs_mean": {"path": ("overlay_comparison", "BASE", "delta_non_event_pairs_mean"), "label": "Mean shift non-events"},
  "delta_min": {"label": "Shift in min (baseline - overlay)"},
  "delta_p50": {"label": "Shift in p50 (baseline - overlay)"},
  "delta_p90": {"label": "Shift in p90 (baseline - overlay)"},
  "normalized_mean_delta": {"label": "Normalized mean shift (delta / baseline mean)"},
  "coverage100": {"path": ("same_day", "coverage100"), "label": "Min/Max Coverage (observations inside envelope)"}, 


  # BIAS
  "PBIAS%": {"path": ("same_day", "PBIAS%"), "label": "PBIAS (%) = 100 * sum(observed - simulated) / sum(observed)"},
  "Bias(obs-pred)": {"path": ("same_day", "Bias(obs-pred)"), "label": "Mean bias (observed - simulated)"},


  # Event contrast (if used)
  "relative_width_event_contrast": {"label": "Event contrast (high / low)"},
  "event_ratio": {"label": "Event ratio (event / non-event)"},
  "relative_width_high": {"label": "Relative width high decile"},
  "relative_width_low": {"label": "Relative width low decile"},

  # __baseline__ ensemble event/nonevent spread
  "bl_event_ratio": {"path": ("overlay_comparison", "__baseline__", "event_ratio"), "label": "Event ratio (ensemble spread)"},
  "bl_RW_event": {"path": ("overlay_comparison", "__baseline__", "relative_width_event"), "label": "Ensemble RW (event days)"},
  "bl_RW_nonevent": {"path": ("overlay_comparison", "__baseline__", "relative_width_nonevent"), "label": "Ensemble RW (non-event days)"},
  "bl_median_W_event": {"path": ("overlay_comparison", "__baseline__", "median_W_event"), "label": "Ensemble median W (event days)"},
  "bl_median_W_nonevent": {"path": ("overlay_comparison", "__baseline__", "median_W_nonevent"), "label": "Ensemble median W (non-event days)"},
  "bl_mean_W_event": {"path": ("overlay_comparison", "__baseline__", "mean_W_event"), "label": "Ensemble mean W (event days)"},
  "bl_mean_W_nonevent": {"path": ("overlay_comparison", "__baseline__", "mean_W_nonevent"), "label": "Ensemble mean W (non-event days)"},
  "bl_p90_W_event": {"path": ("overlay_comparison", "__baseline__", "p90_W_event"), "label": "Ensemble p90 W (event days)"},
  "bl_p90_W_nonevent": {"path": ("overlay_comparison", "__baseline__", "p90_W_nonevent"), "label": "Ensemble p90 W (non-event days)"},

  # Absolute width event ratio
  "bl_abs_W_event_ratio": {"path": ("overlay_comparison", "__baseline__", "abs_width_event_ratio"), "label": "Absolute width event ratio (ensemble)"},

  # Mean-based relative width
  "bl_RW_mean_all": {"path": ("overlay_comparison", "__baseline__", "relative_width_mean"), "label": "Mean relative width (all days)"},
  "bl_RW_mean_event": {"path": ("overlay_comparison", "__baseline__", "relative_width_mean_event"), "label": "Mean relative width (event days)"},
  "bl_RW_mean_nonevent": {"path": ("overlay_comparison", "__baseline__", "relative_width_mean_nonevent"), "label": "Mean relative width (non-event days)"},

  # (max-min)/median metrics
  "bl_median_maxmin_over_median": {"path": ("overlay_comparison", "__baseline__", "median_maxmin_over_median"), "label": "Median (max-min/median) (all days)"},
  "bl_mean_maxmin_over_median": {"path": ("overlay_comparison", "__baseline__", "mean_maxmin_over_median"), "label": "Mean (max-min/median) (all days)"},
  "bl_median_maxmin_over_median_event": {"path": ("overlay_comparison", "__baseline__", "median_maxmin_over_median_event"), "label": "Median (max-min/median) (event days)"},
  "bl_median_maxmin_over_median_nonevent": {"path": ("overlay_comparison", "__baseline__", "median_maxmin_over_median_nonevent"), "label": "Median (max-min/median) (non-event days)"},
  "bl_mean_maxmin_over_median_event": {"path": ("overlay_comparison", "__baseline__", "mean_maxmin_over_median_event"), "label": "Mean (max-min/median) (event days)"},
  "bl_mean_maxmin_over_median_nonevent": {"path": ("overlay_comparison", "__baseline__", "mean_maxmin_over_median_nonevent"), "label": "Mean (max-min/median) (non-event days)"},

}

### helper functions to convert data according to mappings

In [88]:
# Backwards-compatible flat view if you still want a technical-name -> label lookup elsewhere.
semantic_label = {
    technical_name: spec["label"]
    for technical_name, spec in semantic_label_map.items()
}

def _is_scalar_metric_value(value):
    """Return True for JSON scalar metric values that can be stored in the example dict."""
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def _normalise_export_spec(export_spec):
    """
    Accept one export item in one of two forms:
      - {"path": ..., "scenario": ..., "variable": ...}
      - (path, scenario, variable)
    
    Variable can be omitted from the dict if the JSON file contains
    dashboard_state.variable, but scenario must always be supplied because it is
    not part of the current export payload.
    """
    if isinstance(export_spec, dict):
        path = export_spec.get("path")
        scenario = export_spec.get("scenario")
        variable = export_spec.get("variable")
    elif isinstance(export_spec, (tuple, list)) and len(export_spec) == 3:
        path, scenario, variable = export_spec
    else:
        raise TypeError(
            "Each export spec must be either a dict with keys 'path', 'scenario', 'variable' "
            "or a 3-item tuple/list of (path, scenario, variable)."
        )

    if not path:
        raise ValueError(f"Invalid export spec {export_spec!r}: missing file path.")
    if not scenario:
        raise ValueError(
            f"Invalid export spec {export_spec!r}: missing scenario. "
            "Scenario is not stored in the current JSON export, so it must be provided explicitly."
        )

    return Path(path), str(scenario), variable


def _load_stat_export(export_spec, *, verbose=True):
    """Load one JSON export file and return its payload plus resolved identifiers."""
    path, scenario, variable = _normalise_export_spec(export_spec)
    if verbose:
        print(f"Loading stats export: {path}")

    if not path.exists():
        raise FileNotFoundError(f"Stats export file does not exist: {path}")

    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)

    if not isinstance(payload, dict):
        raise ValueError(f"Stats export {path} does not contain a top-level JSON object.")

    dashboard_state = payload.get("metadata", {}).get("dashboard_state") or payload.get("dashboard_state") or {}
    if variable is None:
        variable = dashboard_state.get("variable")

    if not variable:
        raise ValueError(
            f"Could not determine variable for {path}. "
            "Provide it explicitly in the export spec or ensure the payload has dashboard_state.variable."
        )

    stats = payload.get("stats")
    if not isinstance(stats, dict):
        raise ValueError(f"Stats export {path} is missing a 'stats' dictionary.")

    if verbose:
        print(f"  scenario={scenario}, variable={variable}")
    return {
        "path": path,
        "scenario": str(scenario),
        "variable": str(variable),
        "payload": payload,
        "stats": stats,
    }


def _normalise_semantic_entry(technical_name, entry):
    """Normalise one semantic_label_map entry to {'label': ..., 'path': ...}."""
    if isinstance(entry, str):
        spec = {"label": entry, "path": None}
    elif isinstance(entry, dict):
        label = entry.get("label")
        if not label or not isinstance(label, str):
            raise ValueError(
                f"semantic_label_map[{technical_name!r}] must define a string 'label'."
            )
        path = entry.get("path")
        if path is not None:
            if not isinstance(path, (tuple, list)) or not path or not all(isinstance(part, str) for part in path):
                raise ValueError(
                    f"semantic_label_map[{technical_name!r}]['path'] must be a non-empty tuple/list of strings."
                )
            path = tuple(path)
        spec = {"label": label, "path": path}
    else:
        raise TypeError(
            f"semantic_label_map[{technical_name!r}] must be either a label string or a dict with 'label' and optional 'path'."
        )

    return spec


def _normalise_semantic_label_map(semantic_label_map):
    """Return a validated semantic_label_map with uniform dict entries."""
    normalised = {}
    reverse_semantic_label = {}
    duplicate_labels = {}

    for technical_name, entry in semantic_label_map.items():
        spec = _normalise_semantic_entry(technical_name, entry)
        label = spec["label"]
        if label in reverse_semantic_label and reverse_semantic_label[label] != technical_name:
            duplicate_labels.setdefault(label, set()).update([reverse_semantic_label[label], technical_name])
        reverse_semantic_label[label] = technical_name
        normalised[technical_name] = spec

    if duplicate_labels:
        raise ValueError(
            "semantic_label_map contains duplicate labels mapped to different technical names: "
            f"{ {label: sorted(names) for label, names in duplicate_labels.items()} }"
        )

    return normalised


def _find_metric_matches(node, technical_name, path=()):
    """
    Recursively search the exported stats dictionary for a technical metric key.

    Returns a list of (path_tuple, value) matches so ambiguous metrics can be
    reported clearly instead of silently choosing the wrong number.
    """
    matches = []

    if isinstance(node, dict):
        for key, value in node.items():
            next_path = path + (str(key),)
            if key == technical_name and _is_scalar_metric_value(value):
                matches.append((next_path, value))
            matches.extend(_find_metric_matches(value, technical_name, next_path))

    elif isinstance(node, list):
        for index, value in enumerate(node):
            matches.extend(_find_metric_matches(value, technical_name, path + (str(index),)))

    return matches


def _extract_metric_value(stats_dict, technical_name, explicit_path=None, *, verbose=True):
    """Extract one metric value from the exported stats dictionary."""
    if explicit_path is not None:
        explicit_path = tuple(explicit_path)
        current = stats_dict
        for part in explicit_path:
            if not isinstance(current, dict) or part not in current:
                if verbose:
                    print(f"    explicit path {explicit_path} for '{technical_name}' not found, returning NaN")
                return float('nan')
            current = current[part]
        if current is None:
            if verbose:
                print(f"    explicit path {explicit_path} for '{technical_name}' is null, treating as 0")
            return 0
        if not _is_scalar_metric_value(current):
            raise ValueError(
                f"Explicit path {explicit_path!r} for '{technical_name}' does not resolve to a numeric scalar."
            )
        if verbose:
            print(f"    using semantic_label_map path {explicit_path} -> {current}")
        return current

    matches = _find_metric_matches(stats_dict, technical_name)
    if not matches:
        if verbose:
            print(f"    metric '{technical_name}' not found anywhere in the stats export, returning NaN")
        return float('nan')
    if len(matches) > 1:
        pretty_paths = [" -> ".join(path) for path, _ in matches]
        raise ValueError(
            f"Metric '{technical_name}' matched more than one location in the stats export: {pretty_paths}. "
            "Add a 'path' entry to semantic_label_map for this metric."
        )

    metric_path, metric_value = matches[0]
    if verbose:
        print(f"    found {technical_name} at {' -> '.join(metric_path)} = {metric_value}")
    return metric_value


def _format_metric_value(value, fmt, technical_name):
    """Render one numeric value with the metric format string."""
    try:
        return fmt.format(value)
    except Exception as exc:
        raise ValueError(
            f"Could not format value {value!r} for metric '{technical_name}' with format {fmt!r}."
        ) from exc


def _build_requested_metric_specs(semantic_labels, semantic_label_map, format_map):
    """Resolve requested labels to technical names, formats, and optional paths."""
    reverse_semantic_label = {
        spec["label"]: technical_name
        for technical_name, spec in semantic_label_map.items()
    }

    requested_specs = []
    for requested_label in semantic_labels:
        if requested_label in reverse_semantic_label:
            technical_name = reverse_semantic_label[requested_label]
            output_label = requested_label
        elif requested_label in semantic_label_map:
            technical_name = requested_label
            output_label = semantic_label_map[requested_label]["label"]
        else:
            available = sorted(reverse_semantic_label.keys())
            raise KeyError(
                f"Unknown semantic label or technical name: {requested_label!r}. "
                f"Available semantic labels include: {available}"
            )

        if technical_name not in format_map:
            raise KeyError(
                f"No format string found in format_val for technical metric '{technical_name}'. "
                "Add it to format_val before building the metrics dictionary."
            )

        requested_specs.append(
            {
                "technical_name": technical_name,
                "output_label": output_label,
                "format": format_map[technical_name],
                "path": semantic_label_map[technical_name].get("path"),
            }
        )

    return requested_specs


def build_metrics_dict_from_stat_exports(
    export_specs,
    semantic_labels,
    *,
    semantic_label_map=None,
    format_map=None,
    scenario_order=None,
    variable_order=None,
    include_technical_names=False,
    format_mode="metadata",
    verbose=True,
):
    """
    Build a dict with the same structure as the earlier manual 'metrics' blocks from a
    list of dashboard JSON exports.

    Parameters
    ----------
    export_specs : list
        One item per scenario-variable combination. Recommended form:
            {"path": <json_path>, "scenario": "WASTE", "variable": "TP"}
        A tuple/list of (path, scenario, variable) also works.

    semantic_labels : list[str]
        Labels you want in the output dictionary, using the labels defined in
        semantic_label_map, for example:
            ["Median width", "Coverage", "Mean Î”"]
        Technical keys are also accepted directly.

    semantic_label_map : dict, optional
        Mapping of technical metric names to config dicts with:
            - 'label': display label used in semantic_labels
            - 'path' : optional explicit path inside the exported stats dict
        A plain string value is also accepted as shorthand for {'label': <string>}.

    format_map : dict, optional
        Mapping of technical metric names to format strings. If a technical name
        has no format entry, the function raises a helpful error.

    scenario_order / variable_order : list, optional
        If omitted, the function uses the order in which scenarios and variables
        appear in export_specs.

    include_technical_names : bool, default False
        If True, each output metric also gets a '__technical_name' field.

    format_mode : {'metadata', 'values'}, default 'metadata'
        'metadata' keeps numeric values and adds '__format'.
        'values' applies the format string to each extracted value and omits '__format'.

    verbose : bool, default True
        Print progress information while reading files and extracting metrics.

    Returns
    -------
    dict
        A dictionary shaped like:
            {
                "metrics": {
                    "Median width": {
                        "__format": "{:.1f}",
                        "WASTE": {"TP": 12.8, "TN": 17.6},
                        ...
                    },
                    ...
                }
            }
    """
    if semantic_label_map is None:
        semantic_label_map = globals()["semantic_label_map"]
    semantic_label_map = _normalise_semantic_label_map(semantic_label_map)

    if format_map is None:
        format_map = format_val

    valid_format_modes = {"metadata", "values"}
    if format_mode not in valid_format_modes:
        raise ValueError(
            f"format_mode must be one of {sorted(valid_format_modes)}, got {format_mode!r}."
        )

    if not export_specs:
        raise ValueError("export_specs is empty. Provide one export spec per scenario-variable combination.")
    if not semantic_labels:
        raise ValueError("semantic_labels is empty. Provide at least one label to extract.")

    loaded_exports = []
    seen_scenarios = []
    seen_variables = []

    if verbose:
        print(f"Preparing to read {len(export_specs)} stats export files...")

    for export_spec in export_specs:
        loaded = _load_stat_export(export_spec, verbose=verbose)
        loaded_exports.append(loaded)
        if loaded["scenario"] not in seen_scenarios:
            seen_scenarios.append(loaded["scenario"])
        if loaded["variable"] not in seen_variables:
            seen_variables.append(loaded["variable"])

    scenario_order = list(scenario_order) if scenario_order is not None else seen_scenarios
    variable_order = list(variable_order) if variable_order is not None else seen_variables

    if verbose:
        print(f"Scenarios discovered: {scenario_order}")
        print(f"Variables discovered: {variable_order}")

    requested_specs = _build_requested_metric_specs(
        semantic_labels,
        semantic_label_map,
        format_map,
    )

    result = {"metrics": {}}
    for spec in requested_specs:
        output_label = spec["output_label"]
        metric_block = {}
        if format_mode == "metadata":
            metric_block["__format"] = spec["format"]
        if include_technical_names:
            metric_block["__technical_name"] = spec["technical_name"]
        for scenario in scenario_order:
            metric_block[scenario] = {}
        result["metrics"][output_label] = metric_block

    for loaded in loaded_exports:
        scenario = loaded["scenario"]
        variable = loaded["variable"]
        stats_dict = loaded["stats"]

        if scenario not in scenario_order:
            raise ValueError(
                f"Scenario '{scenario}' from {loaded['path']} is not present in scenario_order {scenario_order}."
            )
        if variable not in variable_order:
            raise ValueError(
                f"Variable '{variable}' from {loaded['path']} is not present in variable_order {variable_order}."
            )

        if verbose:
            print(f"Reading metrics for scenario={scenario}, variable={variable}")

        for spec in requested_specs:
            raw_value = _extract_metric_value(
                stats_dict,
                spec["technical_name"],
                explicit_path=spec["path"],
                verbose=verbose,
            )
            if format_mode == "values":
                final_value = _format_metric_value(
                    raw_value,
                    spec["format"],
                    spec["technical_name"],
                )
            else:
                final_value = raw_value
            result["metrics"][spec["output_label"]][scenario][variable] = final_value

    missing_entries = []
    for output_label in result["metrics"]:
        for scenario in scenario_order:
            for variable in variable_order:
                if variable not in result["metrics"][output_label][scenario]:
                    missing_entries.append((output_label, scenario, variable))

    if missing_entries:
        preview = ", ".join([f"{label}/{scenario}/{variable}" for label, scenario, variable in missing_entries[:8]])
        suffix = " ..." if len(missing_entries) > 8 else ""
        raise ValueError(
            "The assembled metrics dictionary is incomplete. Missing entries: "
            f"{preview}{suffix}"
        )

    if verbose:
        print("Finished building metrics dictionary.")
        print(f"Built {len(result['metrics'])} metrics across {len(scenario_order)} scenarios and {len(variable_order)} variables.")

    return result

## Different CHART layouts

### Horizontal stacking of metrics

In [89]:
import math
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

def plot_nested_sensitivity_bars(
    stats_data,
    stats_order=None,
    variables_order=None,       # list of variable names; None -> auto-discover from stats_data
    scenarios_order=None,       # list of scenario names; None -> auto-discover from stats_data
    title="Sensitivity metrics (nested bar chart)",
    y_label="Value",
    figsize=(14, 7),
    abbreviations=None,
    value_format=None,          # callable: (v) or (v, stat) -> str
    annotate_decimals=1,        # used only if neither per-metric nor format_map nor value_format is provided
    format_map=None,            # optional: dict {metric_key: format_string}
    bar_alpha=0.85,
    annotate_rotation=45,
    scenario_label_rotation=0,
    scenario_spacing=0.06,
    bar_width=0.12,
    metric_group_padding="auto",
    min_metric_group_padding=0.14,
    max_metric_group_padding=0.34,
    metric_padding_per_char=0.010,
    metric_padding_reference_chars=16,
    bottom_margin_for_labels=0.24,
    stat_label_map=None,        # optional: {metric_key: display_label}
    annot_fs=16,                # 2x  default (8)
    scenario_fs=16,             # 2x  default (8)
    tick_fs=20,                 # 2x  default (10)
    label_fs=22,                # 2x  default (11)
    title_fs=24,                # 2x  default (12)
    subtitle=None,
    subtitle_fs=None,
    subtitle_y=0.005,
    legend_fs=20,               # 2x  default (10)
    label_offset_scale=0.03,
    label_separation_scale=0.08,
    label_x_nudge_scale=0.22,
    top_margin_scale=0.22,
    inside_legend_top_margin_scale=0.72,
    scenario_label_y_scale=0.08,
    edge_bar_padding=0.75,
    legend_outside=False,
    legend_loc="upper right",
    legend_bbox=(0.98, 0.98),
    tight_layout_rect=(0.0, 0.02, 0.98, 0.98),

    annotation_formatter=None,   # optional callable: (value, stat, scenario, variable, default_text) -> str | None

    info_box_lines=None,          # list of strings to display in an info box on the chart
    info_box_x=0.875,             # horizontal position in axes coords (0-1); 0.875 = centre of 4th quarter
    info_box_y=0.60,              # vertical position in axes coords (0-1); below the legend area
    info_box_fs=None,             # font size (defaults to legend_fs)
    info_box_title=None,          # optional title line rendered in bold above the values
):
    """
    stats_data["metrics"][<STAT>][SCENARIO][VAR] -> float

    Scenarios and variables are auto-discovered from stats_data["metrics"] when
    scenarios_order / variables_order are not provided, so the function works for
    any variable set (TP/TN, TP/TKN, ...).

    Optional per-metric meta-keys inside each <STAT> block:
      - "__format": format string, e.g., "{:.0%}", "{:.1f}", "{:.2f}x"
      - "__label" : short display label for the x-axis; overrides stat_label_map for this stat
    """
    metrics = stats_data.get("metrics", {})
    if not isinstance(metrics, dict) or not metrics:
        raise ValueError("stats_data must contain a non-empty dict at 'metrics'.")

    _first_block = next(
        (v for v in metrics.values() if isinstance(v, dict)),
        {}
    )
    if scenarios_order is not None:
        scenarios = list(scenarios_order)
    else:
        scenarios = [k for k in _first_block if not k.startswith("__")]

    if variables_order is not None:
        variables = list(variables_order)
    else:
        _first_scenario_data = _first_block.get(scenarios[0], {}) if scenarios else {}
        variables = [k for k in _first_scenario_data if not k.startswith("__")]

    if not scenarios:
        raise ValueError("No scenarios found in stats_data and none provided via scenarios_order.")
    if not variables:
        raise ValueError("No variables found in stats_data and none provided via variables_order.")

    if abbreviations is None:
        abbreviations = {sc: sc for sc in scenarios}

    if value_format is not None and not callable(value_format):
        raise TypeError("value_format must be a callable, e.g. lambda v: '...'. You passed a non-callable.")

    if value_format is None:
        fmt_str_default = "{:." + str(int(max(0, annotate_decimals))) + "f}"
        def _format_value(v, stat=None):
            return fmt_str_default.format(v)
    else:
        _orig_fmt = value_format
        def _format_value(v, stat=None):
            try:
                return _orig_fmt(v)
            except TypeError:
                return _orig_fmt(v, stat)

    def _validate_stat(stat_name, block):
        missing = []
        for sc in scenarios:
            if sc not in block:
                missing.append(f"scenario '{sc}'")
            else:
                for var in variables:
                    if var not in block[sc]:
                        missing.append(f"{sc}->{var!r}")
        if missing:
            raise ValueError(f"Metric '{stat_name}' is missing: {', '.join(missing)}")

    for s, block in metrics.items():
        if not isinstance(block, dict):
            raise ValueError(f"Metric '{s}' must map to a dict.")
        _validate_stat(s, block)

    if stats_order is None:
        stats = list(metrics.keys())
    else:
        if not isinstance(stats_order, (list, tuple)):
            raise TypeError("stats_order must be a list/tuple of metric keys.")
        extra = [s for s in stats_order if s not in metrics]
        if extra:
            raise ValueError(f"stats_order contains unknown metric keys: {extra}")
        stats = list(stats_order)

    stat_labels = []
    for s in stats:
        block = metrics[s]
        if isinstance(block, dict) and "__label" in block:
            stat_labels.append(block["__label"])
        elif isinstance(stat_label_map, dict) and s in stat_label_map:
            stat_labels.append(stat_label_map[s])
        else:
            stat_labels.append(s)

    n_stats = len(stats)
    bars_per_subgroup = len(variables)
    subgroup_width = bars_per_subgroup * bar_width
    group_inner = len(scenarios) * subgroup_width + (len(scenarios) - 1) * scenario_spacing

    if metric_group_padding in (None, "auto"):
        inter_group_padding = []
        for i in range(max(0, n_stats - 1)):
            left_len = len(str(stat_labels[i]))
            right_len = len(str(stat_labels[i + 1]))
            label_pressure = max(left_len, right_len) - metric_padding_reference_chars
            padding = min_metric_group_padding + max(0.0, label_pressure) * metric_padding_per_char * (tick_fs / 20.0)
            inter_group_padding.append(min(max_metric_group_padding, max(min_metric_group_padding, padding)))
    else:
        inter_group_padding = [float(metric_group_padding)] * max(0, n_stats - 1)

    group_left_positions = []
    cursor = 0.0
    for i in range(n_stats):
        group_left_positions.append(cursor)
        if i < n_stats - 1:
            cursor += group_inner + inter_group_padding[i]

    fig, ax = plt.subplots(figsize=figsize)

    _known_colors = {
        "TP": "#1f77b4", "TN": "#ff7f0e", "TKN": "#2ca02c",
        "TON": "#d62728", "PP": "#9467bd",
    }
    var_to_color = {
        v: _known_colors.get(v, plt.cm.tab10((i % 10) / 10))
        for i, v in enumerate(variables)
    }

    all_values = [
        float(metrics[stat][scenario][var])
        for stat in stats
        for scenario in scenarios
        for var in variables
        if not math.isnan(float(metrics[stat][scenario][var]))
    ]
    global_max = max(all_values) if all_values else 1.0
    label_base_offset = label_offset_scale * max(1.0, global_max)
    min_label_gap = label_separation_scale * max(1.0, global_max)
    max_label_y = global_max

    for i, stat in enumerate(stats):
        block = metrics[stat]
        per_metric_fmt = None
        if "__format" in block:
            per_metric_fmt = block["__format"]
        elif isinstance(format_map, dict) and stat in format_map:
            per_metric_fmt = format_map[stat]

        group_left = group_left_positions[i]
        for j, scenario in enumerate(scenarios):
            subgroup_left = group_left + j * (subgroup_width + scenario_spacing)
            subgroup_entries = []

            for k, var in enumerate(variables):
                x = subgroup_left + k * bar_width
                y = float(block[scenario][var])
                if math.isnan(y):
                    continue
                ax.bar(x, y, width=bar_width, color=var_to_color[var], alpha=bar_alpha)

                if isinstance(per_metric_fmt, str):
                    try:
                        label_txt = per_metric_fmt.format(y)
                    except Exception:
                        label_txt = str(y)
                else:
                    label_txt = _format_value(y, stat)

                if annotation_formatter is not None:
                    _af_result = annotation_formatter(y, stat, scenario, var, label_txt)
                    if _af_result is not None:
                        label_txt = str(_af_result)

                subgroup_entries.append({
                    "k": k,
                    "x": x,
                    "y": y,
                    "label": label_txt,
                })

            last_label_y = -math.inf
            for lane, entry in enumerate(sorted(subgroup_entries, key=lambda item: item["y"])):
                label_y = entry["y"] + label_base_offset
                if label_y - last_label_y < min_label_gap:
                    label_y = last_label_y + min_label_gap
                x_nudge = (lane - (len(subgroup_entries) - 1) / 2) * label_x_nudge_scale * bar_width
                ax.text(
                    entry["x"] + x_nudge,
                    label_y,
                    entry["label"],
                    ha="center",
                    va="bottom",
                    rotation=annotate_rotation,
                    fontsize=annot_fs,
                )
                last_label_y = label_y
                max_label_y = max(max_label_y, label_y)

            subgroup_center = subgroup_left + (subgroup_width - bar_width) / 2
            ax.text(
                subgroup_center,
                -scenario_label_y_scale * max(1.0, global_max),
                abbreviations.get(scenario, scenario),
                ha="center",
                va="top",
                rotation=scenario_label_rotation,
                fontsize=scenario_fs,
            )

    xticks = [group_left_positions[i] + group_inner / 2 for i in range(n_stats)]
    ax.set_xticks(xticks)
    ax.set_xticklabels(stat_labels, fontsize=tick_fs)

    ax.set_ylabel(y_label, fontsize=label_fs)
    ax.set_title(title, fontsize=title_fs)
    ax.tick_params(axis="y", labelsize=tick_fs)

    plot_left = 0.0
    plot_right = group_left_positions[-1] + group_inner - bar_width if group_left_positions else 0.0
    edge_pad = edge_bar_padding * bar_width
    ax.set_xlim(plot_left - edge_pad, plot_right + edge_pad)

    handles = [Rectangle((0, 0), 1, 1, color=var_to_color[v]) for v in variables]
    labels = list(variables)
    for full_name, abbr in abbreviations.items():
        handles.append(Line2D([0], [0], color="none"))
        #labels.append(f"{abbr} = {full_name}")
    legend_kwargs = dict(
        title="Variables",
        frameon=True,
        handlelength=1.2,
        fontsize=legend_fs,
        title_fontsize=legend_fs,
    )
    ax.legend(handles, labels, loc=legend_loc, bbox_to_anchor=legend_bbox, **legend_kwargs)

    top_reference = max(global_max, max_label_y)
    top_space_scale = top_margin_scale if legend_outside else max(top_margin_scale, inside_legend_top_margin_scale)
    top_pad = top_space_scale * max(1.0, global_max)
    bottom_pad = bottom_margin_for_labels * max(1.0, global_max)
    ax.set_ylim(bottom=-bottom_pad, top=top_reference + top_pad)

    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
    ax.set_axisbelow(True)

    plt.tight_layout(rect=list(tight_layout_rect))
    if subtitle:
        _sub_fs = subtitle_fs if subtitle_fs is not None else max(10, title_fs - 6)
        fig.text(0.5, subtitle_y, subtitle, ha="center", va="bottom",
                 fontsize=_sub_fs, style="italic", color="#444444")
    # ── optional info box ──
    if info_box_lines:
        from matplotlib.offsetbox import TextArea, VPacker, AnchoredOffsetbox
        _ib_fs = info_box_fs if info_box_fs is not None else legend_fs
        _ib_areas = []
        if info_box_title:
            _ib_areas.append(TextArea(info_box_title, textprops=dict(fontsize=_ib_fs, weight="bold")))
        for _ib_entry in info_box_lines:
            if isinstance(_ib_entry, (list, tuple)) and len(_ib_entry) == 2:
                _ib_txt, _ib_clr = _ib_entry
            else:
                _ib_txt, _ib_clr = str(_ib_entry), "black"
            _ib_areas.append(TextArea(_ib_txt, textprops=dict(fontsize=_ib_fs, color=_ib_clr)))
        _ib_pack = VPacker(children=_ib_areas, pad=0, sep=2, align="left")
        _ib_anchor = AnchoredOffsetbox(
            loc="upper left", child=_ib_pack,
            bbox_to_anchor=(info_box_x, info_box_y),
            bbox_transform=ax.transAxes,
            frameon=True, pad=0.4,
        )
        _ib_anchor.patch.set(facecolor="white", alpha=0.8, edgecolor="gray",
                             boxstyle="round,pad=0.4")
        ax.add_artist(_ib_anchor)

    return fig, ax


### Vertical stacking of metrics

In [90]:
import math
from matplotlib.patches import Rectangle

def _normalise_three_metric_specs(metric_specs):
    """Return ordered metric configs for the simplified comparison plot."""
    if metric_specs is None:
        metric_specs = [
            {"title": "Min/Max Coverage (observations inside envelope)"},
            {"title": "PBIAS (%) = 100 * sum(observed - simulated) / sum(observed)"},
            {"title": "Mean bias (observed - simulated)"},
        ]

    if len(metric_specs) != 3:
        raise ValueError("metric_specs must contain exactly 3 metric definitions.")

    normalised_specs = []
    for spec in metric_specs:
        if isinstance(spec, str):
            spec = {"title": spec}
        elif isinstance(spec, dict) and isinstance(spec.get("title"), str):
            spec = dict(spec)
        else:
            raise TypeError(
                "Each metric spec must be either a label string or a dict containing a string 'title'."
            )

        source_title = spec.get("source_title", spec["title"])
        if not isinstance(source_title, str) or not source_title.strip():
            raise ValueError("Each metric spec 'source_title' must be a non-empty string when provided.")

        sign_multiplier = spec.get("sign_multiplier", 1.0)
        try:
            sign_multiplier = float(sign_multiplier)
        except (TypeError, ValueError) as exc:
            raise ValueError(
                "Each metric spec 'sign_multiplier' must be numeric when provided."
            ) from exc

        spec["source_title"] = source_title
        spec["sign_multiplier"] = sign_multiplier
        normalised_specs.append(spec)

    return normalised_specs


def _resolve_three_metric_y_labels(normalised_specs, default_y_label, y_labels):
    """Return one y-axis label per subplot."""
    if y_labels is None:
        return [default_y_label] * len(normalised_specs)

    if isinstance(y_labels, str):
        return [y_labels] * len(normalised_specs)

    if isinstance(y_labels, dict):
        resolved = []
        for spec in normalised_specs:
            label = y_labels.get(spec["title"], y_labels.get(spec["source_title"], default_y_label))
            resolved.append("" if label is None else str(label))
        return resolved

    if isinstance(y_labels, (list, tuple)):
        if len(y_labels) != len(normalised_specs):
            raise ValueError(
                f"y_labels must have exactly {len(normalised_specs)} entries, got {len(y_labels)}."
            )
        return ["" if label is None else str(label) for label in y_labels]

    raise TypeError("y_labels must be None, a string, a dict, or a list/tuple of labels.")


def _build_implied_observed_metric_block(
    pbias_block,
    mean_bias_block,
    scenarios,
    variables,
    *,
    metric_fmt="{:.2g}",
    min_abs_pbias=None,
):
    """Infer mean observed baseline from mean bias and percent bias.

    Since
        PBIAS = 100 * sum(sim - obs) / sum(obs)
    and
        mean_bias = mean(sim - obs) = sum(sim - obs) / n,
    the implied mean observed baseline is
        mean(obs) = 100 * mean_bias / PBIAS
    for the same aligned sample set.
    """
    implied_block = {"__format": metric_fmt}
    for scenario in scenarios:
        implied_block[scenario] = {}
        for variable in variables:
            pbias_value = float(pbias_block[scenario][variable])
            mean_bias_value = float(mean_bias_block[scenario][variable])
            if (
                not math.isfinite(pbias_value)
                or abs(pbias_value) < 1e-12
                or (min_abs_pbias is not None and abs(pbias_value) < min_abs_pbias)
                or not math.isfinite(mean_bias_value)
            ):
                implied_value = float("nan")
            else:
                implied_value = 100.0 * mean_bias_value / pbias_value
            implied_block[scenario][variable] = implied_value
    return implied_block


def _format_small_value_text(raw_value, default_text, *, threshold=None, decimals=None):
    """Preserve the default format except for small magnitudes that need extra precision."""
    if (
        threshold is None
        or decimals is None
        or decimals < 0
        or not math.isfinite(raw_value)
        or abs(raw_value) >= threshold
    ):
        return default_text

    magnitude_text = f"{abs(raw_value):.{decimals}f}".rstrip("0").rstrip(".")
    if not magnitude_text:
        magnitude_text = "0"

    if raw_value < 0:
        return f"-{magnitude_text}"
    if default_text.startswith("+"):
        return f"+{magnitude_text}"
    return magnitude_text


def _format_implied_observed_annotation(
    implied_value,
    metric_fmt,
    *,
    prefix="obs~",
    fallback_text="obs~n/a",
    small_value_threshold=None,
    small_value_decimals=None,
):
    """Format the implied observed baseline annotation shown inside the mean-bias panel."""
    if not math.isfinite(implied_value):
        return fallback_text

    default_text = metric_fmt.format(implied_value)
    value_text = _format_small_value_text(
        implied_value,
        default_text,
        threshold=small_value_threshold,
        decimals=small_value_decimals,
    )
    return f"{prefix}{value_text}"


def _build_implied_observed_annotation_legend_lines(
    *,
    prefix="obs~",
    fallback_text="obs~n/a",
    min_abs_pbias=None,
):
    """Build default legend text explaining implied observed annotations."""
    prefix_text = str(prefix or "obs~")
    example_prefix = prefix_text if prefix_text.endswith("~") else f"{prefix_text} "
    lines = [f"{example_prefix}x = implied mean observed baseline"]

    if fallback_text:
        fallback_line = f"{fallback_text} = annotation unavailable"
        if min_abs_pbias is not None:
            fallback_line = f"{fallback_text} = hidden when |PBIAS| < {min_abs_pbias:g}%"
        lines.append(fallback_line)
    elif min_abs_pbias is not None:
        lines.append(f"Hidden when |PBIAS| < {min_abs_pbias:g}%")

    return lines


def build_three_metric_stats(
    export_specs,
    metric_specs=None,
    *,
    semantic_label_map=None,
    scenario_order=None,
    variable_order=None,
    include_technical_names=False,
    format_mode="metadata",
    verbose=False,
):
    """Build the stats dict consumed by the simplified comparison plot."""
    normalised_specs = _normalise_three_metric_specs(metric_specs)
    semantic_labels = [spec["source_title"] for spec in normalised_specs]
    raw_stats = build_metrics_dict_from_stat_exports(
        export_specs,
        semantic_labels,
        semantic_label_map=semantic_label_map,
        scenario_order=scenario_order,
        variable_order=variable_order,
        include_technical_names=include_technical_names,
        format_mode=format_mode,
        verbose=verbose,
    )

    transformed_stats = {"metrics": {}}
    for spec in normalised_specs:
        source_block = raw_stats["metrics"][spec["source_title"]]
        metric_block = {}
        for key, value in source_block.items():
            if key.startswith("__"):
                metric_block[key] = value
                continue

            metric_block[key] = {
                variable: float(raw_value) * spec["sign_multiplier"]
                for variable, raw_value in value.items()
            }

        if spec["source_title"] != spec["title"]:
            metric_block["__source_title"] = spec["source_title"]
        if spec["sign_multiplier"] != 1.0:
            metric_block["__sign_multiplier"] = spec["sign_multiplier"]

        transformed_stats["metrics"][spec["title"]] = metric_block

    return transformed_stats


def plot_three_metric_comparison(
    export_specs,
    metric_specs=None,
    *,
    semantic_label_map=None,
    scenario_order=None,
    variable_order=None,
    title="Three metric comparison",
    y_label="Value",
    y_labels=None,
    shared_y_label=None,
    show_individual_y_labels=True,
    shared_y_label_x=0.02,
    shared_y_label_fs=None,
    show_implied_observed_panel=False,
    show_implied_observed_annotations=False,
    show_implied_observed_annotation_legend=False,
    implied_observed_title="Implied mean observed baseline",
    implied_observed_y_label="implied mean observed",
    implied_observed_format="{:.2g}",
    implied_observed_annotation_prefix="obs~",
    implied_observed_annotation_fs=None,
    implied_observed_annotation_offset_points=14,
    implied_observed_annotation_legend_title="~obs annotations",
    implied_observed_annotation_legend_lines=None,
    implied_observed_annotation_legend_fs=None,
    implied_observed_annotation_legend_loc="lower right",
    implied_observed_annotation_legend_bbox=(0.98, 0.24),
    implied_observed_min_abs_pbias=None,
    implied_observed_low_pbias_text="obs~n/a",
    third_metric_small_value_threshold=3.0,
    third_metric_small_value_decimals=2,
    figsize=(12, 9),
    abbreviations=None,
    scenario_short=None,
    var_color=None,
    bar_width=0.18,
    group_padding=0.5,
    bar_alpha=0.85,
    annot_fs=18,
    tick_fs=20,
    title_fs=24,
    subtitle=None,
    subtitle_fs=None,
    subtitle_y=0.005,
    label_fs=22,
    legend_fs=20,
    height_scale=0.85,
    annotate_rotation=35,
    format_mode="metadata",
    verbose=False,
    label_offset_scale=0.03,
    annotation_offset_points=6,
    signed_annotation_offset_points=2,
    top_margin_scale=0.22,
    bottom_margin_scale=0.24,
    title_pad=8,
    subplot_h_pad=1.0,
    positive_bar_h_pad_scale=1.6,
    legend_outside=False,
    legend_loc="upper right",
    legend_bbox=(0.98, 0.98),
    tight_layout_rect=(0.0, 0.02, 0.82, 0.94),
    title_clearance_scale=0.05,
    signed_positive_axis_fraction=0.22,
    signed_negative_axis_fraction=0.22,
):
    """Build three metric blocks from dashboard exports and plot them as stacked subplots."""
    if format_mode != "metadata":
        raise ValueError(
            "plot_three_metric_comparison requires format_mode='metadata' so the plot can keep numeric values "
            "and use '__format' for annotations."
        )

    normalised_specs = _normalise_three_metric_specs(metric_specs)

    if scenario_order is None:
        scenario_order = list(dict.fromkeys(s["scenario"] for s in export_specs))
    if variable_order is None:
        variable_order = list(dict.fromkeys(s["variable"] for s in export_specs))

    stats = build_three_metric_stats(
        export_specs,
        metric_specs=normalised_specs,
        semantic_label_map=semantic_label_map,
        scenario_order=scenario_order,
        variable_order=variable_order,
        format_mode=format_mode,
        verbose=verbose,
    )

    metrics = stats["metrics"]
    scenarios = list(scenario_order)
    variables = list(variable_order)
    resolved_y_labels = _resolve_three_metric_y_labels(normalised_specs, y_label, y_labels)

    plot_entries = [
        {
            "title": spec["title"],
            "metric_block": metrics[spec["title"]],
            "y_label": resolved_y_labels[index],
        }
        for index, spec in enumerate(normalised_specs)
    ]

    implied_metric_block = None
    if show_implied_observed_panel or show_implied_observed_annotations:
        implied_metric_block = _build_implied_observed_metric_block(
            metrics[normalised_specs[1]["title"]],
            metrics[normalised_specs[2]["title"]],
            scenarios,
            variables,
            metric_fmt=implied_observed_format,
            min_abs_pbias=implied_observed_min_abs_pbias,
        )
        stats["metrics"][implied_observed_title] = implied_metric_block

    if show_implied_observed_panel and implied_metric_block is not None:
        plot_entries.append(
            {
                "title": implied_observed_title,
                "metric_block": implied_metric_block,
                "y_label": implied_observed_y_label,
            }
        )

    if scenario_short is None:
        scenario_short = abbreviations or {sc: sc for sc in scenarios}

    if var_color is None:
        known_colors = {"TP": "#1f77b4", "TN": "#ff7f0e", "TKN": "#2ca02c", "TON": "#d62728", "PP": "#9467bd"}
        var_color = {v: known_colors.get(v, plt.cm.tab10((i % 10) / 10)) for i, v in enumerate(variables)}

    subgroup_width = len(variables) * bar_width
    group_width = subgroup_width + group_padding

    def group_left(i_scn):
        return i_scn * group_width

    def bar_x(i_scn, i_var):
        return group_left(i_scn) + i_var * bar_width

    def group_center(i_scn):
        return group_left(i_scn) + (subgroup_width - bar_width) / 2

    panel_count = len(plot_entries)
    effective_figsize = (figsize[0], figsize[1] * panel_count / max(1, len(normalised_specs)))
    fig, axes = plt.subplots(panel_count, 1, figsize=effective_figsize, sharex=True)
    if panel_count == 1:
        axes = [axes]

    positive_content_ratios = []
    implied_metric_fmt = implied_observed_format
    if implied_metric_block is not None:
        implied_metric_fmt = implied_metric_block.get("__format", implied_observed_format)

    for idx, plot_entry in enumerate(plot_entries):
        metric_label = plot_entry["title"]
        metric_block = plot_entry["metric_block"]
        metric_fmt = metric_block.get("__format", "{:.1f}")
        ax = axes[idx]

        raw_values = [float(metric_block[scenario][variable]) for scenario in scenarios for variable in variables]
        scaled_values = [value * height_scale for value in raw_values]

        y_min_raw = min(raw_values)
        y_max_raw = max(raw_values)
        signed = y_min_raw < 0
        if signed:
            ax.axhline(0, color="black", linewidth=0.8)

        annotation_offset_points_effective = signed_annotation_offset_points if signed else annotation_offset_points

        positive_bar_top = max([scaled for raw, scaled in zip(raw_values, scaled_values) if raw >= 0] or [0.0])
        negative_bar_bottom = min([scaled for raw, scaled in zip(raw_values, scaled_values) if raw < 0] or [0.0])

        top_pad = top_margin_scale * max(1.0, y_max_raw, positive_bar_top)
        title_clearance = max(
            title_clearance_scale * max(1.0, y_max_raw, positive_bar_top),
            label_offset_scale * max(1.0, y_max_raw),
        )
        bottom_pad = bottom_margin_scale * max(1.0, abs(min(0.0, y_min_raw, negative_bar_bottom)))

        if idx == 2 and show_implied_observed_annotations:
            top_pad += max(4, implied_observed_annotation_offset_points) * max(1.0, label_offset_scale * max(1.0, y_max_raw))
            bottom_pad += max(4, implied_observed_annotation_offset_points) * max(1.0, label_offset_scale * max(1.0, abs(y_min_raw)))

        needed_positive = max(0.0, positive_bar_top) + top_pad + title_clearance
        needed_negative = abs(min(0.0, negative_bar_bottom)) + bottom_pad

        if signed:
            min_upper_from_fraction = needed_negative * signed_positive_axis_fraction / max(1e-6, 1.0 - signed_positive_axis_fraction)
            min_lower_from_fraction = needed_positive * signed_negative_axis_fraction / max(1e-6, 1.0 - signed_negative_axis_fraction)
            upper_bound = max(needed_positive, min_upper_from_fraction)
            lower_bound = -max(needed_negative, min_lower_from_fraction)
        else:
            upper_bound = needed_positive
            lower_bound = -bottom_pad

        ax.set_ylim(bottom=lower_bound, top=upper_bound)

        axis_span = upper_bound - lower_bound
        positive_content_ratios.append((upper_bound / axis_span) if axis_span else 0.0)

        for i_scn, scenario in enumerate(scenarios):
            for i_var, variable in enumerate(variables):
                raw_value = float(metric_block[scenario][variable])
                scaled_value = raw_value * height_scale
                x = bar_x(i_scn, i_var)
                ax.bar(x, scaled_value, width=bar_width, color=var_color[variable], alpha=bar_alpha)

                annotation_text = _format_small_value_text(
                    raw_value,
                    metric_fmt.format(raw_value),
                    threshold=(third_metric_small_value_threshold if idx == 2 else None),
                    decimals=(third_metric_small_value_decimals if idx == 2 else None),
                )

                if raw_value >= 0:
                    ax.annotate(
                        annotation_text,
                        (x, scaled_value),
                        xytext=(0, annotation_offset_points_effective),
                        textcoords="offset points",
                        ha="center",
                        va="bottom",
                        fontsize=annot_fs,
                        clip_on=False,
                    )
                else:
                    ax.annotate(
                        annotation_text,
                        (x, scaled_value),
                        xytext=(0, -annotation_offset_points_effective),
                        textcoords="offset points",
                        ha="center",
                        va="top",
                        fontsize=annot_fs,
                        clip_on=False,
                    )

                if idx == 2 and show_implied_observed_annotations and implied_metric_block is not None:
                    implied_value = float(implied_metric_block[scenario][variable])
                    implied_text = _format_implied_observed_annotation(
                        implied_value,
                        implied_metric_fmt,
                        prefix=implied_observed_annotation_prefix,
                        fallback_text=implied_observed_low_pbias_text,
                        small_value_threshold=third_metric_small_value_threshold,
                        small_value_decimals=third_metric_small_value_decimals,
                    )
                    implied_fontsize = implied_observed_annotation_fs if implied_observed_annotation_fs is not None else max(8, annot_fs - 3)
                    implied_offset = annotation_offset_points_effective + implied_observed_annotation_offset_points
                    if raw_value >= 0:
                        ax.annotate(
                            implied_text,
                            (x, scaled_value),
                            xytext=(0, implied_offset),
                            textcoords="offset points",
                            ha="center",
                            va="bottom",
                            fontsize=implied_fontsize,
                            color="#555555",
                            clip_on=False,
                        )
                    else:
                        ax.annotate(
                            implied_text,
                            (x, scaled_value),
                            xytext=(0, -implied_offset),
                            textcoords="offset points",
                            ha="center",
                            va="top",
                            fontsize=implied_fontsize,
                            color="#555555",
                            clip_on=False,
                        )

        xticks = [group_center(i) for i in range(len(scenarios))]
        ax.set_xticks(xticks)
        ax.set_xticklabels([scenario_short.get(scenario, scenario) for scenario in scenarios], fontsize=tick_fs, rotation=annotate_rotation)
        ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.5)
        ax.set_axisbelow(True)
        ax.set_title(metric_label, fontsize=title_fs, pad=title_pad)
        ax.set_ylabel(plot_entry["y_label"] if show_individual_y_labels else "", fontsize=label_fs)
        ax.tick_params(axis="y", labelsize=tick_fs)

    handles = [Rectangle((0, 0), 1, 1, color=var_color[variable]) for variable in variables]
    legend_kwargs = dict(title="Variables", fontsize=legend_fs, title_fontsize=legend_fs, frameon=True)
    variables_legend = fig.legend(handles, variables, loc=legend_loc, bbox_to_anchor=legend_bbox, **legend_kwargs)

    if show_implied_observed_annotation_legend and show_implied_observed_annotations:
        annotation_legend_lines = implied_observed_annotation_legend_lines
        if annotation_legend_lines is None:
            annotation_legend_lines = _build_implied_observed_annotation_legend_lines(
                prefix=implied_observed_annotation_prefix,
                fallback_text=implied_observed_low_pbias_text,
                min_abs_pbias=implied_observed_min_abs_pbias,
            )
        annotation_legend_lines = [str(line) for line in annotation_legend_lines if str(line).strip()]
        if annotation_legend_lines:
            annotation_legend_fs = (
                implied_observed_annotation_legend_fs
                if implied_observed_annotation_legend_fs is not None
                else max(8, legend_fs - 2)
            )
            annotation_handles = [Rectangle((0, 0), 1, 1, facecolor="none", edgecolor="none") for _ in annotation_legend_lines]
            fig.add_artist(variables_legend)
            fig.legend(
                annotation_handles,
                annotation_legend_lines,
                loc=implied_observed_annotation_legend_loc,
                bbox_to_anchor=implied_observed_annotation_legend_bbox,
                title=implied_observed_annotation_legend_title,
                fontsize=annotation_legend_fs,
                title_fontsize=annotation_legend_fs,
                frameon=True,
                handlelength=0.0,
                handletextpad=0.0,
                borderaxespad=0.0,
                labelspacing=0.8,
            )

    dynamic_h_pad = subplot_h_pad + positive_bar_h_pad_scale * max(positive_content_ratios[:-1] or [0.0])
    fig.suptitle(title, fontsize=title_fs + 1)
    layout_rect = list(tight_layout_rect)
    if shared_y_label is not None:
        layout_rect[0] = max(layout_rect[0], min(0.12, shared_y_label_x + 0.04))
    plt.tight_layout(rect=layout_rect, h_pad=dynamic_h_pad)
    if shared_y_label is not None:
        fig.supylabel(
            shared_y_label,
            x=shared_y_label_x,
            fontsize=(shared_y_label_fs if shared_y_label_fs is not None else label_fs),
        )
    if subtitle:
        subtitle_font_size = subtitle_fs if subtitle_fs is not None else max(10, title_fs - 6)
        fig.text(
            0.5,
            subtitle_y,
            subtitle,
            ha="center",
            va="bottom",
            fontsize=subtitle_font_size,
            style="italic",
            color="#444444",
        )
    return fig, axes, stats

In [91]:
def _normalise_selector_token(value):
    """Normalise exported variable tokens so mappings stay robust."""
    import re
    
    return re.sub(r"[^0-9a-z]+", "-", str(value).strip().lower()).strip("-")


def _extract_run_and_variable_from_stats_filename(filename):
    """Extract the run number and exported variable token from a stats filename."""
    import re
    
    match = re.search(r"run-(\d+)_var-(.+?)_reach", filename)
    if not match:
        return None, None
    return int(match.group(1)), _normalise_selector_token(match.group(2))


def _normalise_run_to_scenario(run_to_scenario):
    """Validate and normalise the run -> scenario mapping."""
    if not isinstance(run_to_scenario, dict):
        raise TypeError("run_to_scenario must be a dict mapping run_number -> scenario.")

    if not run_to_scenario:
        raise ValueError("run_to_scenario is empty")

    normalised = {}
    for run_id, scenario in run_to_scenario.items():
        if not scenario:
            raise ValueError(f"run_to_scenario[{run_id}] is missing a scenario value.")
        normalised[int(run_id)] = str(scenario)

    return normalised


def _normalise_variable_map(variable_map):
    """Validate and normalise the exported-variable -> output-variable mapping."""
    if not isinstance(variable_map, dict):
        raise TypeError(
            "variable_map must be a dict mapping exported variable tokens to output variable names."
        )

    if not variable_map:
        raise ValueError("variable_map is empty")

    return {
        _normalise_selector_token(export_variable): str(output_variable)
        for export_variable, output_variable in variable_map.items()
    }


def build_export_specs_from_folder(
    folder_path,
    run_to_scenario,
    variable_map,
    *,
    file_pattern="run-*.json",
    strict=False,
    verbose=True,
):
    """
    Generate export specs from a folder of dashboard stats JSON files.
    
    Each filename is parsed as:
        run-<RUN>_var-<EXPORTED_VARIABLE>_reach-...json
    
    The caller provides two flat mappings:
        - run_to_scenario : run number -> scenario
        - variable_map    : exported variable token -> output chart variable
    
    Expected mapping shape
    ----------------------
    run_to_scenario = {
        200: "WASTE",
        201: "SOIL",
        202: "WASTE+SOIL",
    }
    
    variable_map = {
        "tot-pkg": "TP",
        "kjeldahl-outkg": "TKN",
    }
    
    Returns a list of dicts shaped like:
        {"path": <json_path>, "scenario": <scenario>, "variable": <output_variable>}
    """
    folder = Path(folder_path)
    if not folder.is_dir():
        raise NotADirectoryError(f"Folder not found: {folder}")

    normalised_run_to_scenario = _normalise_run_to_scenario(run_to_scenario)
    normalised_variable_map = _normalise_variable_map(variable_map)

    json_files = sorted(folder.glob(file_pattern))
    if not json_files:
        raise FileNotFoundError(f"No files matching '{file_pattern}' found in {folder}")

    if verbose:
        print(f"Found {len(json_files)} JSON files in {folder}")

    export_specs = []
    failures = []

    for json_path in json_files:
        run_id, export_variable = _extract_run_and_variable_from_stats_filename(json_path.name)
        if run_id is None or export_variable is None:
            message = f"Could not parse run/variable from filename: {json_path.name}"
            if strict:
                failures.append(message)
            elif verbose:
                print(f"  Warning: {message}")
            continue

        scenario = normalised_run_to_scenario.get(run_id)
        if scenario is None:
            message = f"Run {run_id} from {json_path.name} is not present in run_to_scenario."
            if strict:
                failures.append(message)
            elif verbose:
                print(f"  Warning: {message}")
            continue

        output_variable = normalised_variable_map.get(export_variable)
        if output_variable is None:
            available = sorted(normalised_variable_map.keys())
            message = (
                f"Export variable '{export_variable}' from {json_path.name} is not mapped. "
                f"Available variable_map keys: {available}"
            )
            if strict:
                failures.append(message)
            elif verbose:
                print(f"  Warning: {message}")
            continue

        export_specs.append({
            "path": str(json_path),
            "scenario": scenario,
            "variable": output_variable,
        })

        if verbose:
            print(
                f"  âœ“ run-{run_id}: export_variable={export_variable}, "
                f"scenario={scenario}, variable={output_variable}"
            )

    matched_runs = {
        _extract_run_and_variable_from_stats_filename(Path(spec["path"]).name)[0]
        for spec in export_specs
    }
    missing_runs = sorted(set(normalised_run_to_scenario) - matched_runs)
    if missing_runs and verbose:
        print(f"  Note: no matched files were exported for runs: {missing_runs}")

    if failures:
        raise ValueError("Could not build export specs from folder:\n- " + "\n- ".join(failures))

    if verbose:
        print(f"\nBuilt {len(export_specs)} export specs")

    return export_specs


In [92]:
import re

TITLE_METADATA_PART_OPTIONS = {
    "folder_label",
    "reach",
    "event_threshold",
    "view",
    "n",
    "measured_nonnum_policy",
    "measured_selection",
}

DEFAULT_TITLE_METADATA_PARTS = [
    "folder_label",
    "reach",
    "event_threshold",
    "view",
    "n",
    "measured_nonnum_policy",
    "measured_selection",
]

CHEMICAL_TITLE_ALIASES = {
    "nitrogeno-kjeldahl": "TKN",
    "kjeldahl": "TKN",
    "nitratos": "Nitrate",
    "nitrato": "Nitrate",
    "nitrogeno-total": "Total N",
    "total-nitrogen": "Total N",
    "fosforo-total": "Total P",
}

EVENT_THRESHOLD_TITLE_ALIASES = {
    "p50": "P50",
    "p75": "P75",
    "p90": "P90",
    "abs": "abs threshold",
}

VIEW_TITLE_ALIASES = {
    "all": "all days",
    "events": "events",
    "non_events": "non-events",
    "non-event": "non-events",
    "non-events": "non-events",
}

MEASURED_POLICY_TITLE_ALIASES = {
    "drop": "nonnum drop",
    "keep": "nonnum keep",
    "coerce": "nonnum coerce",
}


def _normalise_title_label_rules(title_folder_label_map):
    """Accept dict shorthand or a list of rule objects for title labels.

    Mapping a token to ``None`` (dict shorthand) or setting ``"label": None``
    (rule-dict format) suppresses that token entirely â€” no default fallback.

    Rule-dict format also supports:
    - ``"order": N``  â€” controls position in the subtitle
      (lower N appears first; unordered tokens keep their natural path order
      after all ordered ones).
    """
    if title_folder_label_map is None:
        return []

    if isinstance(title_folder_label_map, dict):
        rules = []
        for token, label in title_folder_label_map.items():
            if label is None:
                rules.append({"token": str(token), "label": None})
            else:
                rules.append({"token": str(token), "label": str(label)})
        return rules

    if not isinstance(title_folder_label_map, list):
        raise TypeError(
            "title_folder_label_map must be either a dict or a list of rule dictionaries."
        )

    normalised_rules = []
    for rule in title_folder_label_map:
        if not isinstance(rule, dict):
            raise TypeError("Each title rule must be a dict.")
        label = rule.get("label")
        if label is not None and not isinstance(label, str):
            raise ValueError(
                "Each title rule 'label' must be a string or None (to hide)."
            )

        match_keys = [
            key for key in ("token", "contains", "regex")
            if isinstance(rule.get(key), str) and rule.get(key).strip()
        ]
        if len(match_keys) != 1:
            raise ValueError(
                "Each title rule must define exactly one of 'token', 'contains', or 'regex'."
            )

        normalised_rule = {
            match_keys[0]: rule[match_keys[0]],
            "label": label.strip() if isinstance(label, str) else None,
        }
        if "order" in rule and rule["order"] is not None:
            normalised_rule["order"] = int(rule["order"])
        normalised_rules.append(normalised_rule)

    return normalised_rules


def _match_title_rule(token, title_rules):
    """Return ``(label, order)`` for the first matching rule, or ``None``.

    *label* is ``None`` when the token should be hidden (no fallback).
    *order* is an ``int`` when the rule specifies one, otherwise ``None``.
    """
    for rule in title_rules:
        matched = False
        if "token" in rule and token == rule["token"]:
            matched = True
        elif "contains" in rule and rule["contains"] in token:
            matched = True
        elif "regex" in rule and re.search(rule["regex"], token):
            matched = True
        if matched:
            return rule["label"], rule.get("order")
    return None


def _prettify_title_token(token):
    """Convert a folder token into a short human-readable label."""
    token = str(token).strip()
    if not token:
        return None

    if token.startswith("runs_"):
        run_numbers = re.findall(r"\d+", token)
        if run_numbers:
            return "runs " + "/".join(run_numbers)

    pretty = token.replace("__", " ").replace("_", " ").replace("-", " ")
    pretty = re.sub(r"\s+", " ", pretty).strip()
    return pretty.title() if pretty else None


def _extract_folder_title_tokens(batch_folder):
    """Extract meaningful folder tokens from a dashboard stats batch folder path."""
    folder = Path(batch_folder)
    parts = list(folder.parts)
    dashboard_index = next(
        (index for index, part in enumerate(parts) if str(part).lower() == "dashboard_stats"),
        None,
    )

    relevant_parts = parts[dashboard_index + 1 :] if dashboard_index is not None else parts[-3:]
    if not relevant_parts:
        return []

    tokens = [str(part) for part in relevant_parts[:-1] if str(part).strip()]
    final_part = str(relevant_parts[-1]).strip()
    if final_part:
        tokens.append(final_part.split("__", 1)[0])
    return tokens


def _build_folder_title_label(batch_folder, title_folder_label_map=None):
    """Build a semantic label from folder path segments and optional translation rules.

    Supports ``order`` for explicit positioning and ``None`` labels to hide tokens.
    """
    title_rules = _normalise_title_label_rules(title_folder_label_map)
    _INF = float("inf")
    entries = []  # list of (order, natural_index, label)
    for natural_index, token in enumerate(_extract_folder_title_tokens(batch_folder)):
        match = _match_title_rule(token, title_rules)
        if match is not None:
            label, order = match
            if label is None:
                continue  # explicitly hidden â€” no fallback
        else:
            label = _prettify_title_token(token)
            order = None
        if label and label not in [e[2] for e in entries]:
            entries.append((order if order is not None else _INF, natural_index, label))
    entries.sort(key=lambda e: (e[0], e[1]))
    labels = [e[2] for e in entries]
    return ", ".join(labels) if labels else Path(batch_folder).name


def _translate_title_value(value, aliases, fallback=None):
    """Translate a metadata value with alias lookup and a simple fallback."""
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None

    token = _normalise_selector_token(text)
    if token in aliases:
        return aliases[token]
    if callable(fallback):
        return fallback(text)
    return text


def _translate_measured_chemical(chemical_name):
    """Return a compact chemistry label for measured selection summaries."""
    return _translate_title_value(
        chemical_name,
        CHEMICAL_TITLE_ALIASES,
        fallback=lambda text: text.title(),
    )


def _summarise_measured_selection(measured_selection):
    """Summarise enabled measured chemistry and station choices for chart titles."""
    if not isinstance(measured_selection, dict):
        return None

    enabled_entries = [
        entry for entry in measured_selection.values()
        if isinstance(entry, dict) and entry.get("enabled")
    ]
    if not enabled_entries:
        return None

    chemistry_labels = []
    station_sets = []
    for entry in enabled_entries:
        chemical = _translate_measured_chemical(entry.get("chemical"))
        if chemical and chemical not in chemistry_labels:
            chemistry_labels.append(chemical)

        stations = tuple(
            str(station).strip()
            for station in (entry.get("stations") or [])
            if str(station).strip()
        )
        if stations and stations not in station_sets:
            station_sets.append(stations)

    if not chemistry_labels:
        return None

    summary = f"measured: {' + '.join(chemistry_labels)}"
    if len(station_sets) == 1:
        station_label = "Station" if len(station_sets[0]) == 1 else "Stations"
        summary = f"{summary}; {station_label}: {' + '.join(station_sets[0])}"
    elif len(station_sets) > 1:
        summary = f"{summary}; Stations: mixed"

    return summary


def _normalise_title_metadata_parts(title_metadata_parts=None):
    """Validate title metadata part selection while preserving order."""
    parts = DEFAULT_TITLE_METADATA_PARTS if title_metadata_parts is None else list(title_metadata_parts)
    invalid_parts = [part for part in parts if part not in TITLE_METADATA_PART_OPTIONS]
    if invalid_parts:
        raise ValueError(
            f"Unknown title metadata parts: {invalid_parts}. Valid options are {sorted(TITLE_METADATA_PART_OPTIONS)}."
        )

    deduplicated = []
    for part in parts:
        if part not in deduplicated:
            deduplicated.append(part)
    return deduplicated


def _format_title_sample_size(value):
    """Format a numeric sample size compactly for titles."""
    if value is None:
        return None
    if isinstance(value, (int, float)) and float(value).is_integer():
        return f"n={int(value)}"
    return f"n={value}"


def build_semantic_chart_title(
    batch_folder,
    export_specs,
    chart_descriptor,
    *,
    title_metadata_parts=None,
    title_folder_label_map=None,
    verbose=False,
):
    """Build a short semantic chart title from folder tokens and one example JSON export."""
    selected_parts = _normalise_title_metadata_parts(title_metadata_parts)
    if not export_specs:
        return chart_descriptor

    loaded_export = _load_stat_export(export_specs[0], verbose=False)
    payload = loaded_export["payload"]
    metadata = payload.get("metadata") or {}
    dashboard_state = metadata.get("dashboard_state") or payload.get("dashboard_state") or {}
    stats_dict = loaded_export["stats"]
    event_context = stats_dict.get("event_context") or {}

    extracted_values = {
        "folder_label": _build_folder_title_label(
            batch_folder,
            title_folder_label_map=title_folder_label_map,
        ),
        "reach": f"R{dashboard_state.get('reach')}" if dashboard_state.get("reach") is not None else None,
        "event_threshold": _translate_title_value(
            dashboard_state.get("event_threshold"),
            EVENT_THRESHOLD_TITLE_ALIASES,
            fallback=lambda text: text.upper(),
        ),
        "view": _translate_title_value(
            event_context.get("view") or dashboard_state.get("event_view"),
            VIEW_TITLE_ALIASES,
            fallback=lambda text: text.replace("_", " "),
        ),
        "n": _format_title_sample_size(stats_dict.get("n")),
        "measured_nonnum_policy": _translate_title_value(
            dashboard_state.get("measured_nonnum_policy"),
            MEASURED_POLICY_TITLE_ALIASES,
            fallback=lambda text: f"nonnum {text}",
        ),
        "measured_selection": _summarise_measured_selection(
            dashboard_state.get("measured_selection")
        ),
    }

    folder_label = None
    compact_parts = []
    measured_selection_summary = None
    for part_name in selected_parts:
        value = extracted_values.get(part_name)
        if not value:
            continue
        if part_name == "folder_label":
            folder_label = value
        elif part_name == "measured_selection":
            measured_selection_summary = value
        else:
            compact_parts.append(value)

    subtitle_segments = []
    if folder_label:
        subtitle_segments.append(folder_label)
    if compact_parts:
        subtitle_segments.append(", ".join(compact_parts))
    if measured_selection_summary:
        subtitle_segments.append(measured_selection_summary)

    title = chart_descriptor if chart_descriptor else ""
    subtitle = " | ".join(seg for seg in subtitle_segments if seg) if subtitle_segments else None
    if verbose:
        print(f"Semantic title for {Path(batch_folder).name}: {title}")
        print(f"  subtitle: {subtitle}")
    return title, subtitle


In [93]:




#How the auto-subtitle construction works and an annotated example showin how to control it:


# â”€â”€â”€ ORDER â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# title_metadata_parts is a list â€” its ORDER determines the subtitle
# segment order, and its PRESENCE controls which parts appear at all.
# Valid keys: "folder_label", "reach", "event_threshold", "view",
#             "n", "measured_nonnum_policy", "measured_selection"

title_metadata_parts = [
    "folder_label",           # represents interpolation error
    "view",                   # represents event vs non-event vs all days
    "event_threshold",        # defines event definition (e.g. P75, abs)
    "measured_selection",     # summarises which chemicals and stations were included in the measured data for this chart (e.g. "measured: TKN + Nitrate; Station: 30304")
    "n",                      # number of measurements considere by above filter and mapping
    "reach",                  # represents the reach being analysed (e.g. 13)
    #"measured_nonnum_policy", 
]
# Subtitle template:  folder_label | view, reach, event_threshold, n | measured_selection


# â”€â”€â”€ SEMANTIC MAPPINGS â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Each extracted value is translated through its own alias dict.
# You can override any of them before the chart cells run.

# 1) folder tokens â†’ human labels  (used by "folder_label"); order will be folowed; set None to skip a segment entirely with no fallback
title_folder_label_map = {
    # exact-match shorthand  (token: label)
    "runs_187_200_202":      "Soil interpol. err.: rel. median",
    "runs_205_200_204":      "Soil interpol. err.: abs. median",
    "runs_189_200_203":      "Soil interpol. err.: rel. RMSE",
    "runs_188_200_201":      "Soil interpol. err.: P75",
    "runs_153_155_157":      "Original TFM (rel. RMSE)",
    "drop_mdl":              "drop-below-DL",
    "half_mdl":              "half-DL",
    "non-event-days":        "non-event-days",
    "event-days":            "event-days",
}
# or use a list of rule-dicts for contains / regex matching:
# title_folder_label_map = [
#     {"contains": "188",  "label": "P75 soil uncertainty"},
#     {"regex":    r"^runs_153", "label": "original TFM"},
# ]

# 2) event threshold tokens â†’ labels  (used by "event_threshold")
EVENT_THRESHOLD_TITLE_ALIASES = {
    "p50": "event thresh.: P50",
    "p75": "event thresh.: P75",
    "p90": "event thresh.: P90",
    "abs": "event thresh.: abs",
}

# 3) view tokens â†’ labels  (used by "view")
VIEW_TITLE_ALIASES = {
    "all":        "all days",
    "events":     "event days",
    "non_events": "non-event days",
    "non-event":  "non-event days",
    "non-events": "non-event days",
}

# 4) measured nonnum policy tokens â†’ labels  (used by "measured_nonnum_policy")
MEASURED_POLICY_TITLE_ALIASES = {
    "drop":   "nonnum drop",
    "keep":   "nonnum keep",
    "coerce": "nonnum coerce",
}

# 5) chemistry tokens â†’ labels  (used inside "measured_selection")
CHEMICAL_TITLE_ALIASES = {
    "nitrogeno-kjeldahl": "TKN",
    "nitratos":           "Nitrate",
    "nitrogeno-total":    "TN",
    "fosforo-total":      "TP",
}




# The resulting subtitle is: `folder_label | compact_part1, compact_part2, â€¦ | measured_selection`

# So to **reorder**, just rearrange `title_metadata_parts`. To **rename** what appears, modify the corresponding alias dict or `title_folder_label_map`. To **hide** a part, remove it from the list.

**How the pieces compose at runtime:**

| `extracted_values` key | Source | Alias dict | Subtitle placement |
|---|---|---|---|
| `folder_label` | folder path tokens | `title_folder_label_map` | own segment (first `\|` block) |
| `reach` | `dashboard_state["reach"]` | formatted as `R{n}` | compact group |
| `event_threshold` | `dashboard_state["event_threshold"]` | `EVENT_THRESHOLD_TITLE_ALIASES` | compact group |
| `view` | `event_context["view"]` | `VIEW_TITLE_ALIASES` | compact group |
| `n` | `stats["n"]` | formatted as `n={v}` | compact group |
| `measured_nonnum_policy` | `dashboard_state["measured_nonnum_policy"]` | `MEASURED_POLICY_TITLE_ALIASES` | compact group |
| `measured_selection` | `dashboard_state["measured_selection"]` | `CHEMICAL_TITLE_ALIASES` (internal) | own segment (last `\|` block) |

# Charts

In [94]:

#format only coverage as percent:
value_format_map = {
  "Coverage fraction": (lambda v: f"{v*100:.1f}%")
}

In [95]:

# Extends semantic_label_map with explicit metric paths needed for width-by-event metrics.
# These metrics appear at multiple locations in the JSON, so explicit paths prevent ambiguity.
example_semantic_label_map = {
    **semantic_label_map,
    # RW_high / RW_low now point to __baseline__ (ensemble spread) via semantic_label_map.
    # No overrides needed here unless you want to switch to BASE overlay deviations.
}


In [96]:
# Example usage: declare one or more batch export folders to process in one pass.
# Each folder will generate the full set of three charts.

batch_folders = [
    
    # 188 - csv_p75_absolute_relative_error
        # TN + TP (half_mdl, all days)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\runs_188_200_201__vars_tot_nkg_tot_pkg"),
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\conc\runs_188_200_201__vars_tot_nkg_tot_pkg"),
        # TN + TP (half_mdl, non_event)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\non-event\conc\runs_188_200_201__vars_tot_nkg_tot_pkg"),
        # TN + TP (half_mdl, event)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\events\runs_188_200_201__vars_tot_nkg_tot_pkg"),
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\events\conc\runs_188_200_201__vars_tot_nkg_tot_pkg"),
       
        # TN + TP (drop_mdl, all days)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\runs_188_200_201__vars_tot_nkg_tot_pkg"),
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\conc\runs_188_200_201__vars_tot_nkg_tot_pkg"),
        # TN + TP (drop_mdl, non-event)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\non-event-days\runs_188_200_201__vars_tot_nkg_tot_pkg"),
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\non-event-days\conc\runs_188_200_201__vars_tot_nkg_tot_pkg"),

    # 205 - csv_absolute_relative_error
        # TN + TP (drop_mdl, all days)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\runs_205_200_204__vars_tot_nkg_tot_pkg"),
         # TN + TP (half_mdl, all days)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\runs_205_200_204__vars_tot_nkg_tot_pkg"),
        # TN + TP (drop_mdl, non-event days)
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\non-event-days\runs_205_200_204__vars_tot_nkg_tot_pkg"),


    # 187 - csv_median_relative_error
         # TN + TP (half_mdl, all days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\conc\runs_187_200_202__vars_tot_nkg_tot_pkg"),
        # TN + TP (drop_mdl, all days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\runs_187_200_202__vars_tot_nkg_tot_pkg"),
        # TN + TP (drop_mdl, non-event days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\non-event-days\runs_187_200_202__vars_tot_nkg_tot_pkg"),

    # 189 - rmse_rel_center
         # TN + TP (drop_mdl, all days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\runs_189_200_203__vars_tot_nkg_tot_pkg"),
         # TN + TP (half_mdl, all days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\half_mdl\runs_189_200_203__vars_tot_nkg_tot_pkg"),
        # TN + TP (drop_mdl, non-event days)
    #Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\drop_mdl\non-event-days\runs_189_200_203__vars_tot_nkg_tot_pkg"),
       
       
        # orig TFM
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\runs_153_155_157__vars_tot_nkg_tot_pkg"),

]

run_to_scenario = {
    # shared WASTE baseline
    200: "WASTE",

    # 188 group
    188: "SOIL",
    201: "WASTE+SOIL",

    # 205 group
    205: "SOIL",
    204: "WASTE+SOIL",

    # 187 group
    187: "SOIL",
    202: "WASTE+SOIL",

    # 189 group
    189: "SOIL",
    203: "WASTE+SOIL",

    # orig TFM
    157: "WASTE",
    153: "SOIL",
    155: "WASTE+SOIL",
}

variable_map = {
    "tot-nkg": "TN",
    "tot-pkg": "TP",
    "kjeldahl-outkg": "TKN",
}

chart_scenario_order = ["WASTE", "SOIL", "WASTE+SOIL"]
chart_variable_order = ["TP", "TN"]
chart_abbreviations = {"WASTE": "WASTE", "SOIL": "SOIL", "WASTE+SOIL": "W+S"}

save_individual_chart_images = False
saved_chart_root = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\sensitivity_chart_exports")
saved_chart_dpi = 220


# # â”€â”€ Title builder configuration â”€â”€
# # Which metadata parts to include in each chart title (in order).
# title_metadata_parts = [
#     "folder_label",
#     "reach",
#     "event_threshold",
#     "view",
#     "n",
#     "measured_nonnum_policy",
#     "measured_selection",
# ]

# # Map raw folder tokens to human-readable labels.
# # Accepts a simple dict (token â†’ label) or a list of rule dicts.
# title_folder_label_map = {
#     "drop_mdl": "drop-MDL",
#     "half_mdl": "half-MDL",
#     "non-event-days": "non-events",
#     "runs_188_200_201": "soil uncertainty = P75",
#     "runs_205_200_204": "soil uncertainty = |median relative error|",
#     "runs_187_200_202": "soil uncertainty = median relative error",
#     "runs_189_200_203": "soil uncertainty = RMSE rel. center",
#     "runs_153_155_157": "original TFM",
# }

print(f"Configured {len(batch_folders)} batch folder(s).")
for folder in batch_folders:
    print(f"  - {folder.name}")
print(f"Title metadata parts: {title_metadata_parts}")


Configured 13 batch folder(s).
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_205_200_204__vars_tot_nkg_tot_pkg
  - runs_205_200_204__vars_tot_nkg_tot_pkg
  - runs_205_200_204__vars_tot_nkg_tot_pkg
  - runs_153_155_157__vars_tot_nkg_tot_pkg
Title metadata parts: ['folder_label', 'view', 'event_threshold', 'measured_selection', 'n', 'reach']


In [97]:
# ── Batch-folder builder ──────────────────────────────────────────────────────
# Mirrors the folder hierarchy produced by the combinatorial wrapper in
# notebook 03: batch_folder_exports / {chm} / {policy} / {view} / {mode} / runs_…
#
# Pass an *ordered* dict of axis → values.  The dict iteration order controls
# which axis varies fastest in the returned list (last key = innermost loop).
# The on-disk hierarchy is always chm/policy/view/mode regardless of dict order.

import itertools, re

BATCH_EXPORT_ROOT = Path(
    r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM"
    r"\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM"
    r"\config\outputs\dashboard_stats\batch_folder_exports"
)

# Maps global_chm_uncert → sorted (run_a, run_b, run_c) newest run numbers
SCENARIO_RUNS = {
    "csv_p75_absolute_relative_error":    (188, 200, 201),
    "csv_median_absolute_relative_error": (204, 200, 205),
    "csv_median_relative_error":          (187, 200, 202),
    "rmse_rel_center":                    (189, 200, 203),
}

_POLICY_FOLDER_MAP = {"half_mdl": "half_mdl", "drop": "drop_mdl"}


def build_batch_folders(
    axes: dict,
    *,
    batch_root: Path = BATCH_EXPORT_ROOT,
    scenario_runs: dict = SCENARIO_RUNS,
    variables: tuple = ("TOT_Nkg", "TOT_Pkg"),
) -> list:
    """Build an ordered list of batch-export folder Paths.

    Parameters
    ----------
    axes : dict[str, list[str]]
        Keys from {"global_chm_uncert", "policy", "event_view", "compare_mode"}.
        Values are the desired values for that axis.
        Dict iteration order controls which axis varies fastest (last = innermost).
    batch_root : Path
        Root of the batch_folder_exports tree.
    scenario_runs : dict
        Maps global_chm_uncert key → tuple of sorted run numbers.
    variables : tuple
        SWAT variable names used to build the leaf folder token.

    Returns
    -------
    list[Path]
    """
    def _sanitize(v):
        return re.sub(r"[^0-9A-Za-z]+", "_", v).strip("_").lower() or "var"

    vars_token = "_".join(_sanitize(v) for v in variables)

    def _runs_folder(chm):
        nums = sorted(scenario_runs[chm])
        return f"runs_{'_'.join(str(n) for n in nums)}__vars_{vars_token}"

    def _policy_folder(p):
        return _POLICY_FOLDER_MAP.get(p.lower(), p.lower())

    axis_keys = list(axes.keys())
    axis_values = [axes[k] for k in axis_keys]

    result = []
    for combo in itertools.product(*axis_values):
        combo_dict = dict(zip(axis_keys, combo))
        chm    = combo_dict["global_chm_uncert"]
        policy = combo_dict["policy"]
        view   = combo_dict["event_view"]
        mode   = combo_dict["compare_mode"]

        path = (
            batch_root
            / chm
            / _policy_folder(policy)
            / view
            / mode
            / _runs_folder(chm)
        )
        result.append(path)

    return result


# ── Build batch_folders from the new hierarchy ────────────────────────────────
# Dict order → list order: policy varies slowest, compare_mode varies fastest.

batch_folders = build_batch_folders({
    "global_chm_uncert":  ["csv_p75_absolute_relative_error", "csv_median_absolute_relative_error"],
    "compare_mode":       ["load", "conc"],
    "policy":             ["half_mdl", "drop"],
    "event_view":         ["all", "events", "non_events"],
})

# ── Append legacy / non-hierarchy folders if needed ───────────────────────────

# batch_folders += [
#     # orig TFM
#     Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\runs_153_155_157__vars_tot_nkg_tot_pkg"),
# ]


run_to_scenario = {
    # shared WASTE baseline
    200: "WASTE",

    # 188 group
    188: "SOIL",
    201: "WASTE+SOIL",

    # 205 group
    205: "SOIL",
    204: "WASTE+SOIL",

    # 187 group
    187: "SOIL",
    202: "WASTE+SOIL",

    # 189 group
    189: "SOIL",
    203: "WASTE+SOIL",

    # orig TFM
    157: "WASTE",
    153: "SOIL",
    155: "WASTE+SOIL",
}

variable_map = {
    "tot-nkg": "TN",
    "tot-pkg": "TP",
    "kjeldahl-outkg": "TKN",
}

chart_scenario_order = ["WASTE", "SOIL", "WASTE+SOIL"]
chart_variable_order = ["TP", "TN"]
chart_abbreviations = {"WASTE": "WASTE", "SOIL": "SOIL", "WASTE+SOIL": "W+S"}

save_individual_chart_images = False
saved_chart_root = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\outputs\dashboard_stats\sensitivity_chart_exports")
saved_chart_dpi = 220


# # ── Title builder configuration ──
# # Which metadata parts to include in each chart title (in order).
# title_metadata_parts = [
#     "folder_label",
#     "reach",
#     "event_threshold",
#     "view",
#     "n",
#     "measured_nonnum_policy",
#     "measured_selection",
# ]

# # Map raw folder tokens to human-readable labels.
# title_folder_label_map = {
#     "drop_mdl": "drop-MDL",
#     "half_mdl": "half-MDL",
#     "non-event-days": "non-events",
#     "runs_188_200_201": "soil uncertainty = P75",
#     "runs_205_200_204": "soil uncertainty = |median relative error|",
#     "runs_187_200_202": "soil uncertainty = median relative error",
#     "runs_189_200_203": "soil uncertainty = RMSE rel. center",
#     "runs_153_155_157": "original TFM",
# }

print(f"Configured {len(batch_folders)} batch folder(s).")
for folder in batch_folders:
    print(f"  - {folder.name}")
print(f"Title metadata parts: {title_metadata_parts}")

Configured 24 batch folder(s).
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_188_200_201__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__vars_tot_nkg_tot_pkg
  - runs_200_204_205__v

In [98]:
from contextlib import redirect_stdout
import io

def _ordered_present_values(actual_values, preferred_order=None):
    """Keep discovered order unless a preferred order is provided."""
    actual_values = list(actual_values)
    if preferred_order is None:
        return actual_values

    preferred_present = [value for value in preferred_order if value in actual_values]
    extras = [value for value in actual_values if value not in preferred_present]
    return preferred_present + extras


def _save_chart_figure(fig, folder_label, chart_slug, *, save_enabled=False, output_root=None, dpi=220):
    """Optionally save one chart figure to a per-folder output directory."""
    if not save_enabled:
        return None

    if output_root is None:
        raise ValueError("output_root must be provided when save_enabled=True")

    output_dir = Path(output_root) / folder_label
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{chart_slug}.png"
    fig.savefig(output_path, dpi=dpi, bbox_inches="tight")
    print(f"Saved chart image: {output_path}")
    return output_path


def _capture_stdout(callable_obj, *args, **kwargs):
    """Run a callable and return (result, captured_stdout)."""
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        result = callable_obj(*args, **kwargs)
    return result, buffer.getvalue()


# ── Chart 1a: Median-based relative width (existing) ──
ch1a_semantic_labels = [
    "Model relative width median((p95-p05)/p50)",
    "Relative width (event days)",
    "Relative width (non-event days)",
    "Event ratio (ensemble spread)",
]

# ── Chart 1b: Absolute width (event vs non-event) ──
ch1b_semantic_labels = [
    "Ensemble median W (event days)",
    "Ensemble median W (non-event days)",
    "Absolute width event ratio (ensemble)",
]

# ── Chart 1c: Mean-based relative width ──
ch1c_semantic_labels = [
    "Mean relative width (all days)",
    "Mean relative width (event days)",
    "Mean relative width (non-event days)",
]

# ── Chart 1d: (max-min)/median metrics ──
ch1d_semantic_labels = [
    "Median (max-min/median) (all days)",
    "Mean (max-min/median) (all days)",
    "Median (max-min/median) (event days)",
    "Median (max-min/median) (non-event days)",
    "Mean (max-min/median) (event days)",
    "Mean (max-min/median) (non-event days)",
]

# Combined list for backward-compatibility references
ch1_semantic_labels = ch1a_semantic_labels + ch1b_semantic_labels + ch1c_semantic_labels + ch1d_semantic_labels

ch2_semantic_labels = [
    "Mean shift (vs. BASE)",
    "Median shift (vs. BASE)",
    "Mean shift events",
    "Mean shift non-events",
]


# "delta_mean": {"path": ("overlay_comparison", "BASE", "delta_mean"), "label": "Mean shift (vs. BASE)"},
#"mean_delta_pct": {"path": ("overlay_comparison", "BASE", "delta_mean_pct"), "label": "Percent of mean shift (vs. BASE)"},
#"median_delta": {"path": ("overlay_comparison", "BASE", "median_delta"), "label": "Median shift (vs. BASE)"},
#"median_delta_pct": {"path": ("overlay_comparison", "BASE", "median_delta_pct"), "label": "Median percent shift (vs. BASE)"},


chart_results_by_folder = {}

### Multi-folder chart execution

The next chart cells run each chart once for every folder listed in `batch_folders`.

- Each figure title includes the current folder name.
- If `save_individual_chart_images = True`, each chart is also saved separately per folder.

In [ ]:
chart_1_metrics_by_folder = {}
chart_1_logs = []

# ── Chart configuration for 4 sub-charts ──────────────────────────────────
_chart_1_variants = [
    {
        "slug": "chart_1a_relative_width_median",
        "labels": ch1a_semantic_labels,
        "descriptor": "Relative sensitivity — median((p95−p05)/p50)",
        "y_label": "factor compared to envelope median",
        "stat_label_map": {
            "Model relative width median((p95-p05)/p50)": "Rel. width\n(all days)",
            "Relative width (event days)": "Rel. width\n(events)",
            "Relative width (non-event days)": "Rel. width\n(non-events)",
            "Event ratio (ensemble spread)": "Event ratio\n(rel. width)",
        },
        "figsize": (16, 8),
    },
    {
        "slug": "chart_1b_absolute_width",
        "labels": ch1b_semantic_labels,
        "descriptor": "Absolute envelope width — event vs non-event",
        "y_label": "width (variable units / day)",
        "stat_label_map": {
            "Ensemble median W (event days)": "Abs. width\n(events)",
            "Ensemble median W (non-event days)": "Abs. width\n(non-events)",
            "Absolute width event ratio (ensemble)": "Event ratio\n(abs. width)",
        },
        "figsize": (14, 8),
    },
    {
        "slug": "chart_1c_relative_width_mean",
        "labels": ch1c_semantic_labels,
        "descriptor": "Relative sensitivity — mean-based relative width",
        "y_label": "factor compared to envelope mean",
        "stat_label_map": {
            "Mean relative width (all days)": "Rel. width\n(all days)",
            "Mean relative width (event days)": "Rel. width\n(events)",
            "Mean relative width (non-event days)": "Rel. width\n(non-events)",
        },
        "figsize": (14, 8),
    },
    {
        "slug": "chart_1d_maxmin_over_median",
        "labels": ch1d_semantic_labels,
        "descriptor": "Spread sensitivity — (max−min)/median",
        "y_label": "factor compared to envelope median",
        "stat_label_map": {
            "Median (max-min/median) (all days)": "Median\n(max-min)/med\n(all)",
            "Mean (max-min/median) (all days)": "Mean\n(max-min)/med\n(all)",
            "Median (max-min/median) (event days)": "Median\n(max-min)/med\n(events)",
            "Median (max-min/median) (non-event days)": "Median\n(max-min)/med\n(non-events)",
            "Mean (max-min/median) (event days)": "Mean\n(max-min)/med\n(events)",
            "Mean (max-min/median) (non-event days)": "Mean\n(max-min)/med\n(non-events)",
        },
        "figsize": (20, 8),
    },
]

ch_1_title_metadata_parts = [
    "folder_label",
    "event_threshold",
]

ch_1_title_folder_label_map = {
    "runs_187_200_202":      "Soil interpolation error: relative median",
    "runs_205_200_204":      "Soil interpolation error: absolute median",
    "runs_189_200_203":      "Soil interpolation error: relative RMSE",
    "runs_188_200_201":      "Soil interpolation error: 75th percentile",
    "runs_153_155_157":      "Original TFM (relative RMSE)",
    "drop_mdl":              None,
    "half_mdl":              None,
}

EVENT_THRESHOLD_TITLE_ALIASES = {
    "p50": "event defintion: above 50th flow percentile",
    "p75": "event defintion: above 75th flow percentile",
    "p90": "event defintion: above 90th flow percentile",
    "abs": "event defintion: above absolute flow value",
}

for batch_folder in batch_folders:
    generated_specs, build_log = _capture_stdout(
        build_export_specs_from_folder,
        batch_folder,
        run_to_scenario,
        variable_map,
        verbose=True,
    )

    _ch1_actual_scenarios = list(dict.fromkeys(spec["scenario"] for spec in generated_specs))
    _ch1_actual_variables = list(dict.fromkeys(spec["variable"] for spec in generated_specs))
    _ch1_selected_scenarios = _ordered_present_values(_ch1_actual_scenarios, chart_scenario_order)
    _ch1_selected_variables = _ordered_present_values(_ch1_actual_variables, chart_variable_order)

    # Build metrics for ALL ch1 labels at once (one pass over the JSON files)
    ch_1_auto_metrics = build_metrics_dict_from_stat_exports(
        generated_specs,
        ch1_semantic_labels,
        semantic_label_map=example_semantic_label_map,
        scenario_order=_ch1_selected_scenarios,
        variable_order=_ch1_selected_variables,
        format_mode="metadata",
        verbose=False,
    )

    for variant in _chart_1_variants:
        variant_metrics = {
            "metrics": {
                k: v for k, v in ch_1_auto_metrics["metrics"].items()
                if k in variant["labels"]
            }
        }

        chart_title, chart_subtitle = build_semantic_chart_title(
            batch_folder, generated_specs,
            variant["descriptor"],
            title_metadata_parts=ch_1_title_metadata_parts,
            title_folder_label_map=ch_1_title_folder_label_map,
        )

        fig, ax = plot_nested_sensitivity_bars(
            variant_metrics,
            variables_order=_ch1_selected_variables,
            title=chart_title,
            subtitle=chart_subtitle,
            y_label=variant["y_label"],
            figsize=variant["figsize"],
            scenarios_order=_ch1_selected_scenarios,
            stats_order=variant["labels"],
            stat_label_map=variant["stat_label_map"],
            abbreviations=chart_abbreviations,
            annotate_rotation=60,
            annot_fs=22,
            tick_fs=24,
            label_fs=22,
            title_fs=28,
            subtitle_fs=18,
            legend_fs=26,
            legend_outside=False,
            legend_loc="upper right",
            legend_bbox=(0.98, 0.98),
            tight_layout_rect=(0.0, 0.02, 0.98, 0.98),
            inside_legend_top_margin_scale=0.82,
            edge_bar_padding=0.70,
            subtitle_y=-0.0075,
            scenario_fs=20,
            scenario_label_rotation=0,
            scenario_spacing=0.06,
            bottom_margin_for_labels=0.14,
            label_offset_scale=0.03,
            label_separation_scale=0.08,
            label_x_nudge_scale=0.09,
            scenario_label_y_scale=0.02,
        )
        plt.show()

        saved_path, save_log = _capture_stdout(
            _save_chart_figure,
            fig,
            batch_folder.name,
            variant["slug"],
            save_enabled=save_individual_chart_images,
            output_root=saved_chart_root,
            dpi=saved_chart_dpi,
        )
        plt.close(fig)

        chart_results_by_folder.setdefault(batch_folder.name, {})[variant["slug"] + "_metrics"] = variant_metrics
        chart_results_by_folder[batch_folder.name][variant["slug"] + "_saved_path"] = saved_path

        if save_log.strip():
            chart_1_logs.append(save_log.rstrip())

    chart_1_metrics_by_folder[batch_folder.name] = ch_1_auto_metrics
    chart_results_by_folder.setdefault(batch_folder.name, {})["chart_1_metrics"] = ch_1_auto_metrics

    chart_1_logs.append(f"Chart 1 folder: {batch_folder.name}")
    chart_1_logs.append(f"  scenarios: {_ch1_selected_scenarios}")
    chart_1_logs.append(f"  variables: {_ch1_selected_variables}")
    if build_log.strip():
        chart_1_logs.append(build_log.rstrip())
    chart_1_logs.append("")

print("\nFinished chart 1 (a/b/c/d) for all configured folders.")
print("\n".join(line for line in chart_1_logs if line is not None))

In [ ]:
chart_2_metrics_by_folder = {}

chart_2_logs = []

# ── Info-box configuration (external BASE overlay metric shown per variable) ──
ch2_info_box_overlay_stat = "mean"                            # switch to "mean" to display the external BASE mean
_ch2_info_box_metric_options = {
    "median": {
        "metric_name": "overlay_p50",
        "label": "median BASE prediction",
    },
    "mean": {
        "metric_name": "overlay_mean",
        "label": "mean BASE prediction",
    },
}
ch2_info_box_overlay_stat = str(ch2_info_box_overlay_stat).strip().lower()
if ch2_info_box_overlay_stat not in _ch2_info_box_metric_options:
    raise ValueError(
        f"Unsupported ch2_info_box_overlay_stat={ch2_info_box_overlay_stat!r}. "
        f"Choose one of: {sorted(_ch2_info_box_metric_options)}"
    )
_ch2_info_box_metric_cfg = _ch2_info_box_metric_options[ch2_info_box_overlay_stat]
ch2_info_box_prefix = f"{_ch2_info_box_metric_cfg['label']}:"   # text before the numeric value
ch2_info_box_metric_name = _ch2_info_box_metric_cfg["metric_name"]
ch2_info_box_x = 0.025                                           # horizontal position (0–1 in axes coords)
ch2_info_box_y = 0.95                                            # vertical position (0–1 in axes coords)
ch2_info_box_metric_path = ("overlay_comparison", "BASE", ch2_info_box_metric_name)



for batch_folder in batch_folders:

    generated_specs, build_log = _capture_stdout(

        build_export_specs_from_folder,

        batch_folder,

        run_to_scenario,

        variable_map,

        verbose=True,

    )



    _ch2_actual_scenarios = list(dict.fromkeys(spec["scenario"] for spec in generated_specs))

    _ch2_actual_variables = list(dict.fromkeys(spec["variable"] for spec in generated_specs))

    _ch2_selected_scenarios = _ordered_present_values(_ch2_actual_scenarios, chart_scenario_order)

    _ch2_selected_variables = _ordered_present_values(_ch2_actual_variables, chart_variable_order)



    ch_2_auto_metrics = build_metrics_dict_from_stat_exports(

        generated_specs,

        ch2_semantic_labels,

        semantic_label_map=example_semantic_label_map,

        scenario_order=_ch2_selected_scenarios,

        variable_order=_ch2_selected_variables,

        format_mode="metadata",

        verbose=False,

    )

    # Extract the selected external BASE overlay summary for the info box (first JSON per variable)
    _ch2_overlay_reference_value = {}
    for _spec in generated_specs:
        _loaded = _load_stat_export(_spec, verbose=False)
        _var = _loaded["variable"]
        if _var not in _ch2_overlay_reference_value:
            _val = _extract_metric_value(
                _loaded["stats"], ch2_info_box_metric_name,
                explicit_path=ch2_info_box_metric_path,
                verbose=False,
            )
            _ch2_overlay_reference_value[_var] = _val
    _ch2_var_colors = {"TP": "#1f77b4", "TN": "#ff7f0e", "TKN": "#2ca02c", "TON": "#d62728", "PP": "#9467bd"}
    _ch2_info_box_lines = [
        (f"{var}: {ch2_info_box_prefix} {_ch2_overlay_reference_value[var]:.2f}", _ch2_var_colors.get(var, "black"))
        for var in _ch2_selected_variables
        if var in _ch2_overlay_reference_value
    ]



    _ch2_example_export = _load_stat_export(generated_specs[0], verbose=False)

    _ch2_dashboard_state = (

        _ch2_example_export["payload"].get("metadata", {}).get("dashboard_state")

        or _ch2_example_export["payload"].get("dashboard_state")

        or {}

    )

    _ch2_compare_mode = str(_ch2_dashboard_state.get("compare_mode") or "load").strip().lower()

    if _ch2_compare_mode == "conc":

        ch_2_chart_descriptor = "Shift sensitivity compared to BASE scenario (concentrations)"
        ch_2_y_label = "shift vs. BASE (mg/L)"

        def _ch2_annotation_formatter(value, stat, scenario, variable, default_text):
            if variable == "TP":
                return f"{value:+.1f}"
            return default_text

    else:

        ch_2_chart_descriptor = "Shift sensitivity compared to BASE scenario (total loads)"
        ch_2_y_label = "shift vs. BASE (kg/day)"
        _ch2_annotation_formatter = None



    ch_2_title_metadata_parts = [

    "folder_label",           # represents interpolation error and MDL handling policy

    #"view",                   # represents event vs non-event vs all days

    "event_threshold",        # defines event definition (e.g. P75, abs)

    #"measured_selection",     # summarises which chemicals and stations were included in the measured data for this chart (e.g. "measured: TKN + Nitrate; Station: 30304")

    #"n",                      # number of measurements considere by above filter and mapping

    #"reach",                  # represents the reach being analysed (e.g. 13)

    #"measured_nonnum_policy", 

    ]



    ch_2_title_folder_label_map = {

        # exact-match shorthand  (token: label)

        "runs_187_200_202":      "Soil interpolation error: relative median",

        "runs_205_200_204":      "Soil interpolation error: absolute median",

        "runs_189_200_203":      "Soil interpolation error: relative RMSE",

        "runs_188_200_201":      "Soil interpolation error: 75th percentile",

        "runs_153_155_157":      "Original TFM (relative RMSE)",

        "drop_mdl":              None, #"drop-below-DL",

        "half_mdl":              None, #"half-DL",

    }





    EVENT_THRESHOLD_TITLE_ALIASES = {

    "p50": "event defintion: above 50th flow percentile",

    "p75": "event defintion: above 75th flow percentile",

    "p90": "event defintion: above 90th flow percentile",

    "abs": "event defintion: above absolute flow value",

    }       



    chart_title, chart_subtitle = build_semantic_chart_title(

        batch_folder, generated_specs,

        ch_2_chart_descriptor,

        title_metadata_parts=ch_2_title_metadata_parts,

        title_folder_label_map=ch_2_title_folder_label_map,

    )



    fig, ax = plot_nested_sensitivity_bars(

        ch_2_auto_metrics,

        title=chart_title,

        subtitle=chart_subtitle,

        variables_order=_ch2_selected_variables,

        scenarios_order=_ch2_selected_scenarios,

        abbreviations=chart_abbreviations,

        y_label=ch_2_y_label,

        figsize=(20, 8),

        annotate_rotation=60,

        scenario_label_rotation=0,


        annot_fs=22,

        scenario_fs=24,

        tick_fs=24,

        label_fs=22,

        title_fs=28,

        legend_fs=26,

        legend_outside=False,

        legend_loc="upper right",

        legend_bbox=(0.98, 0.98),

        tight_layout_rect=(0.0, 0.02, 0.98, 0.98),

        inside_legend_top_margin_scale=0.90,

        edge_bar_padding=0.90,

        label_separation_scale=0.02,
        label_offset_scale=0.01,
        label_x_nudge_scale=0.29,

        subtitle_y=-0.0075, # the smaller the lower the subtitle (can be negative to go below the plot area

        bottom_margin_for_labels=0.34,  # for scenario labels

        annotation_formatter=_ch2_annotation_formatter,

        info_box_lines=_ch2_info_box_lines,
        info_box_x=ch2_info_box_x,
        info_box_y=ch2_info_box_y,

    )

    plt.show()



    saved_path, save_log = _capture_stdout(

        _save_chart_figure,

        fig,

        batch_folder.name,

        "chart_2_shift",

        save_enabled=save_individual_chart_images,

        output_root=saved_chart_root,

        dpi=saved_chart_dpi,

    )

    plt.close(fig)



    chart_2_metrics_by_folder[batch_folder.name] = ch_2_auto_metrics

    chart_results_by_folder.setdefault(batch_folder.name, {})["chart_2_metrics"] = ch_2_auto_metrics

    chart_results_by_folder[batch_folder.name]["chart_2_saved_path"] = saved_path



    chart_2_logs.append(f"Chart 2 folder: {batch_folder.name}")

    chart_2_logs.append(f"  compare_mode: {_ch2_compare_mode}")

    chart_2_logs.append(f"  info box overlay stat: {ch2_info_box_overlay_stat}")

    chart_2_logs.append(f"  scenarios: {_ch2_selected_scenarios}")

    chart_2_logs.append(f"  variables: {_ch2_selected_variables}")

    if build_log.strip():

        chart_2_logs.append(build_log.rstrip())

    if save_log.strip():

        chart_2_logs.append(save_log.rstrip())

    chart_2_logs.append("")



print("\nFinished chart 2 for all configured folders.")

print("\n".join(line for line in chart_2_logs if line is not None))


In [ ]:
chart_3_metrics_by_folder = {}

chart_3_logs = []



# For chart 3:

# - title controls the subplot title you see on the figure

# - source_title points to the exported metric label used to fetch the raw value

# - sign_multiplier lets you flip the displayed sign without changing the export files

three_metric_specs = [

    {

        "title": "Min/Max Coverage (observations inside envelope)",

    },

    {

        "source_title": "PBIAS (%) = 100 * sum(observed - simulated) / sum(observed)",

        "title": "PBIAS (%) = 100 * sum(sim. - obs.) / sum(obs.)",

        "sign_multiplier": -1,

    },

    {

        "source_title": "Mean bias (observed - simulated)",

        "title": "Mean bias (simulated - observed)",

        "sign_multiplier": -1,

    },

]



# Chart 3 y-axis label options:

# - chart_3_y_labels gives one label per subplot, top to bottom

# - chart_3_shared_y_label adds one label spanning all plotted subplots

# - set chart_3_show_individual_y_labels = False if you only want the shared label

chart_3_y_labels = [

    "coverage fraction",

    "percent bias (%)",

    "mean bias (units of variable)",

]

chart_3_shared_y_label = None

chart_3_show_individual_y_labels = True

chart_3_shared_y_label_x = 0.02



# Implied observed baseline diagnostic for chart 3:

# - set chart_3_show_implied_observed_panel = True to restore the separate fourth panel

# - set chart_3_show_implied_observed_annotations = True to show compact labels in the mean-bias panel

# - values are suppressed as chart_3_implied_observed_low_pbias_text when |PBIAS| is below the threshold

chart_3_show_implied_observed_panel = False

chart_3_show_implied_observed_annotations = True

chart_3_implied_observed_title = "Implied mean observed baseline"

chart_3_implied_observed_y_label = "implied mean obs."

chart_3_implied_observed_format = "{:.4g}"

chart_3_implied_observed_annotation_prefix = "obs~"

chart_3_implied_observed_annotation_offset_points = 18

chart_3_implied_observed_min_abs_pbias = 5.0

chart_3_implied_observed_low_pbias_text = "obs~n/a"

chart_3_third_metric_small_value_threshold = 3.0

chart_3_third_metric_small_value_decimals = 1

chart_3_show_implied_observed_annotation_legend = True

chart_3_implied_observed_annotation_legend_title = "~obs ="

chart_3_implied_observed_annotation_legend_lines = [

    "mean of ",

    "measured",

    "values",

    #"n/a: |PBIAS| < 5%",

]

chart_3_implied_observed_annotation_legend_fs = 18

chart_3_implied_observed_annotation_legend_loc = "lower left"

chart_3_implied_observed_annotation_legend_bbox = (0.835, 0.07)



three_metric_semantic_label_map = {

    **semantic_label_map,

    **example_semantic_label_map,

    "coverage100": {

        "label": "Min/Max Coverage (observations inside envelope)",

        "path": ("same_day", "coverage100"),

    },

    "PBIAS%": {

        "label": "PBIAS (%) = 100 * sum(observed - simulated) / sum(observed)",

        "path": ("same_day", "PBIAS%"),

    },

    "Bias(obs-pred)": {

        "label": "Mean bias (observed - simulated)",

        "path": ("same_day", "Bias(obs-pred)"),

    },

}





for batch_folder in batch_folders:

    generated_specs, build_log = _capture_stdout(

        build_export_specs_from_folder,

        batch_folder,

        run_to_scenario,

        variable_map,

        verbose=True,

    )



    _ch3_actual_scenarios = list(dict.fromkeys(spec["scenario"] for spec in generated_specs))

    _ch3_actual_variables = list(dict.fromkeys(spec["variable"] for spec in generated_specs))

    _ch3_selected_scenarios = _ordered_present_values(_ch3_actual_scenarios, chart_scenario_order)

    _ch3_selected_variables = _ordered_present_values(_ch3_actual_variables, chart_variable_order)



    _ch3_example_export = _load_stat_export(generated_specs[0], verbose=False)

    _ch3_dashboard_state = (

        _ch3_example_export["payload"].get("metadata", {}).get("dashboard_state")

        or _ch3_example_export["payload"].get("dashboard_state")

        or {}

    )

    _ch3_compare_mode = str(_ch3_dashboard_state.get("compare_mode") or "load").strip().lower()

    _ch3_y_labels = list(chart_3_y_labels)

    if _ch3_compare_mode == "conc":

        ch_3_chart_descriptor = "** Simulated vs. measured (concentrations) **"

        _ch3_y_labels[2] = "mean bias (mg/L)"

    else:

        ch_3_chart_descriptor = "** Simulated vs. measured (total loads) **"

        _ch3_y_labels[2] = "mean bias (kg/day)"



    ch_3_title_metadata_parts = [

    "folder_label",           # represents interpolation error and MDL handling policy

    "view",                   # represents event vs non-event vs all days

    "event_threshold",        # defines event definition (e.g. P75, abs)

    #"measured_selection",     # summarises which chemicals and stations were included in the measured data for this chart (e.g. "measured: TKN + Nitrate; Station: 30304")

    #"n",                      # number of measurements considere by above filter and mapping

    "reach",                  # represents the reach being analysed (e.g. 13)

    "measured_nonnum_policy", 

    ]



    ch_3_title_folder_label_map = {

        # exact-match shorthand  (token: label)

        "runs_187_200_202":      "Soil interpolation error: relative median",

        "runs_205_200_204":      "Soil interpolation error: absolute median",

        "runs_189_200_203":      "Soil interpolation error: relative RMSE",

        "runs_188_200_201":      "Soil interpolation error: 75th percentile",

        "runs_153_155_157":      "Original TFM (relative RMSE)",

        "drop_mdl":              None, #"drop-below-DL",

        "half_mdl":              None, #"half-DL",

    }





    EVENT_THRESHOLD_TITLE_ALIASES = {

    "p50": "event defintion: above 50th flow percentile",

    "p75": "event defintion: above 75th flow percentile",

    "p90": "event defintion: above 90th flow percentile",

    "abs": "event defintion: above absolute flow value",

    }       



    chart_title, chart_subtitle = build_semantic_chart_title(

        batch_folder, generated_specs,

        ch_3_chart_descriptor,

        title_metadata_parts=ch_3_title_metadata_parts,

        title_folder_label_map=ch_3_title_folder_label_map,

    )





    fig, ax, three_metric_stats = plot_three_metric_comparison(

        generated_specs,

        metric_specs=three_metric_specs,

        semantic_label_map=three_metric_semantic_label_map,

        variable_order=_ch3_selected_variables,

        scenario_order=_ch3_selected_scenarios,

        title=chart_title,

        subtitle=chart_subtitle,

        y_label="compared to msrmnts",

        y_labels=_ch3_y_labels,

        shared_y_label=chart_3_shared_y_label,

        show_individual_y_labels=chart_3_show_individual_y_labels,

        shared_y_label_x=chart_3_shared_y_label_x,

        show_implied_observed_panel=chart_3_show_implied_observed_panel,

        show_implied_observed_annotations=chart_3_show_implied_observed_annotations,

        implied_observed_title=chart_3_implied_observed_title,

        implied_observed_y_label=chart_3_implied_observed_y_label,

        implied_observed_format=chart_3_implied_observed_format,

        implied_observed_annotation_prefix=chart_3_implied_observed_annotation_prefix,

        implied_observed_annotation_offset_points=chart_3_implied_observed_annotation_offset_points,

        implied_observed_min_abs_pbias=chart_3_implied_observed_min_abs_pbias,

        implied_observed_low_pbias_text=chart_3_implied_observed_low_pbias_text,

        third_metric_small_value_threshold=chart_3_third_metric_small_value_threshold,

        third_metric_small_value_decimals=chart_3_third_metric_small_value_decimals,



        show_implied_observed_annotation_legend=chart_3_show_implied_observed_annotation_legend,

        implied_observed_annotation_legend_title=chart_3_implied_observed_annotation_legend_title,

        implied_observed_annotation_legend_lines=chart_3_implied_observed_annotation_legend_lines,

        implied_observed_annotation_legend_fs=chart_3_implied_observed_annotation_legend_fs,

        implied_observed_annotation_legend_loc=chart_3_implied_observed_annotation_legend_loc,

        implied_observed_annotation_legend_bbox=chart_3_implied_observed_annotation_legend_bbox,



        figsize=(12, 9),

        abbreviations=chart_abbreviations,

        annotate_rotation=60,

        annot_fs=15,

        tick_fs=20,

        title_fs=24,

        subtitle_fs=10,

        label_fs=20,

        legend_fs=20,

        title_pad=12,

        subplot_h_pad=1.1,

        positive_bar_h_pad_scale=1.8,

        legend_outside=False,

        legend_loc="upper right",

        legend_bbox=(0.96, 0.835),

        tight_layout_rect=(0.0, 0.02, 0.82, 0.94),

        verbose=False,

        signed_positive_axis_fraction=0.32,

        signed_negative_axis_fraction=0.32,

        label_offset_scale=0.03,

        annotation_offset_points=6,

        signed_annotation_offset_points=2,

        top_margin_scale=0.22,

        height_scale=0.85,

    )

    plt.show()



    saved_path, save_log = _capture_stdout(

        _save_chart_figure,

        fig,

        batch_folder.name,

        "chart_3_three_metric",

        save_enabled=save_individual_chart_images,

        output_root=saved_chart_root,

        dpi=saved_chart_dpi,

    )

    plt.close(fig)



    chart_3_metrics_by_folder[batch_folder.name] = three_metric_stats

    chart_results_by_folder.setdefault(batch_folder.name, {})["chart_3_metrics"] = three_metric_stats

    chart_results_by_folder[batch_folder.name]["chart_3_saved_path"] = saved_path



    chart_3_logs.append(f"Chart 3 folder: {batch_folder.name}")

    chart_3_logs.append(f"  compare_mode: {_ch3_compare_mode}")

    chart_3_logs.append(f"  scenarios: {_ch3_selected_scenarios}")

    chart_3_logs.append(f"  variables: {_ch3_selected_variables}")

    if build_log.strip():

        chart_3_logs.append(build_log.rstrip())

    if save_log.strip():

        chart_3_logs.append(save_log.rstrip())

    chart_3_logs.append("")



print("\nFinished chart 3 for all configured folders.")

print("\n".join(line for line in chart_3_logs if line is not None))
